# SUPERESTELA — Stuff+ y Location+ (versión limpia)

**Orden del notebook**

1. Lectura de datos
2. Normalización
3. Event type (`set_event_type`)
4. Run values (`runValues.csv` → `delta_run_exp`)
5. Funciones de plot
6. Targets auxiliares (como los tenías)
7. xRunValue: clases y tabla de valores
8. Features derivadas (espejo de lateralidad, fastball principal, densidad del aire)
9. Entrenamiento Stuff+ por tipo de pitcheo
10. Entrenamiento Location+ por tipo de pitcheo
11. Validación
12. Visualizaciones
13. Rankings de pitchers

**Decisiones de diseño (por qué cambió respecto a tus notebooks)**

- **Dos targets distintos.** Stuff+ se entrena *solo con swings* y sin ubicación. Location+ se entrena con *todos* los pitcheos y solo con ubicación, conteo y lateralidad.
- **Probabilidades multiclase calibradas** (sin `scale_pos_weight`, que las distorsiona) × **run value de tu CSV** por evento y conteo. Ya no hay pesos inventados ni `run_value` real dentro del xRV.
- **Validación por pitcher** (GroupKFold): cada pitcher se predice con modelos que no lo vieron.
- **Altura:** `density_index` como feature, y un Stuff+ "ref" evaluado a densidad de nivel del mar.

## 1. Instalación y lectura de datos

In [ ]:
%pip install pandas pyarrow xgboost scikit-learn matplotlib seaborn

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

# ------------------------------- CONFIG -------------------------------------
DATA_PATH        = "stuff_model_df.parquet"   # <-- tu dataset (LMB, o MLB2024.parquet para la prueba de MLB)
RUN_VALUES_PATH  = "runValues.csv"
MODELS_DIR       = "models_versionNueva"
RANDOM_STATE     = 42

GROUP_COL        = "pitcher_anon_id"
GAME_COL         = "game_anon_id"
PITCH_TYPE_COL   = "AutoPitchType"
ALTITUDE_COL     = "altitude_category"

PITCH_TYPE_ALIASES   = {}                                  # ej. {"TwoSeamFastBall": "Sinker"}
EXCLUDE_PITCH_TYPES  = ["Other", "Knuckleball", "Undefined"]
MIN_SWINGS_PER_TYPE  = 3000      # mínimo de swings para entrenar Stuff+ de un tipo
MIN_PITCHES_LOC      = 8000      # mínimo de pitcheos para entrenar Location+ de un tipo

FAST_MODE            = False     # True = 3 folds y menos árboles (para probar rápido)
N_FOLDS              = 3 if FAST_MODE else 5
N_ROUNDS             = 120 if FAST_MODE else 250

USE_DENSITY          = True      # density_index como feature de Stuff+
USE_FASTBALL_DIFFS   = True      # diferenciales vs fastball principal del pitcher
STUFF_RV_MODE        = "neutral" # "neutral": valor por clase sin importar el conteo | "by_count": usa el conteo del pitcheo

In [ ]:
df = pd.read_parquet(DATA_PATH)
print("Filas:", f"{len(df):,}", "| Columnas:", df.shape[1])

REQUIRED_COLS = [
    "year", "PitchUID", GAME_COL, GROUP_COL, "PitcherThrows", "BatterSide", PITCH_TYPE_COL,
    "RelSpeed", "SpinRate", "SpinAxis", "RelHeight", "RelSide", "Extension",
    "InducedVertBreak", "HorzBreak", "VertRelAngle", "HorzRelAngle",
    "PlateLocHeight", "PlateLocSide", "PitchCall", "play_result", "Balls", "Strikes",
]
OPTIONAL_COLS = [ALTITUDE_COL, "vx0", "vy0", "vz0", "ax0", "ay0", "az0",
                 "hit_type", "ExitSpeed", "Angle", "is_swing", "is_whiff"]
missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}")
print("Opcionales ausentes:", [c for c in OPTIONAL_COLS if c not in df.columns] or "ninguna")

## 2. Normalización de datos

In [ ]:
def normalize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia tipos de dato mixtos ANTES de construir targets o entrenar.

    1) Fuerza a float las columnas numéricas que llegaron como 'object' (ej. EffectiveVelo).
    2) Convierte columnas bandera 0/1 a booleano real.
    """
    df = df.copy()
    if "EffectiveVelo" in df.columns:
        df["EffectiveVelo"] = pd.to_numeric(df["EffectiveVelo"], errors="coerce")
    for col in df.columns:
        if df[col].nunique() == 2 and set(df[col].dropna().unique()) == {0.0, 1.0}:
            df[col] = df[col].astype(bool)
    return df


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Ajustes comunes que necesitan todos los pasos siguientes."""
    df = df.copy().reset_index(drop=True)
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    for c in ["Balls", "Strikes", "PlateLocHeight", "PlateLocSide"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    if PITCH_TYPE_ALIASES:
        df[PITCH_TYPE_COL] = df[PITCH_TYPE_COL].replace(PITCH_TYPE_ALIASES)
    if ALTITUDE_COL not in df.columns:
        df[ALTITUDE_COL] = "No Altitude"
    df["valid_hand"] = df["PitcherThrows"].isin(["Left", "Right"]) & df["BatterSide"].isin(["Left", "Right"])
    if "in_strike_zone" not in df.columns:   # zona aproximada (pies); sustituye por la tuya si ya la tienes
        df["in_strike_zone"] = ((df["PlateLocSide"].abs() <= 0.83)
                                & df["PlateLocHeight"].between(1.55, 3.5)).astype(int)
    return df


df = normalize_columns(normalize_dtypes(df))
print("Pitch types:", df[PITCH_TYPE_COL].value_counts().to_dict())
print("Manos inválidas:", (~df["valid_hand"]).sum())

## 3. Event type

Cada pitcheo se clasifica en un evento discreto. La lógica es la tuya (`classify_event`), vectorizada para que no tarde.
Se agregó un **rescate**: si `PitchCall` es `BallCalled`, `BallinDirt`, `StrikeCalled`, `StrikeSwinging` o `HitByPitch` y tu regla dejó el evento vacío, se usa el `PitchCall`. El notebook imprime cuántos pitcheos se rescataron.

In [ ]:
FOUL_CALLS = ["FoulBall", "FoulBallNotFieldable", "FoulBallFieldable"]
RESCUE_MAP = {"BallCalled": "BallCalled", "BallinDirt": "BallCalled", "StrikeCalled": "StrikeCalled",
              "StrikeSwinging": "StrikeSwinging", "HitByPitch": "HitByPitch"}


def classify_event(row):
    """Versión fila a fila (tu lógica original). Queda como referencia y para pruebas."""
    pitch_call = row.get("PitchCall")
    play_result = row.get("play_result")
    if play_result == "NeutralPlay":
        if pitch_call in FOUL_CALLS:
            return "FoulBall"
        elif pitch_call == "StrikeCalled":
            return "StrikeCalled"
        else:
            return None
    elif play_result == "Strikeout":
        return pitch_call
    elif play_result == "Error":
        return "Out"
    elif play_result == "FieldersChoice":
        return "Out"
    elif play_result == "Sacrifice":
        return "Out"
    elif play_result == "BallinDirt":
        return "BallCalled"
    elif play_result == "FoulBallFieldable":
        return "FoulBall"
    return play_result


def set_event_type(df: pd.DataFrame, rescue: bool = True) -> pd.DataFrame:
    """Misma lógica que classify_event, vectorizada (+ rescate opcional)."""
    df = df.copy()
    pc = df["PitchCall"].to_numpy(dtype=object)
    pr = df["play_result"].to_numpy(dtype=object)
    isin = lambda arr, vals: pd.Series(arr).isin(vals).to_numpy()

    out = pr.copy()
    done = np.zeros(len(df), dtype=bool)

    def put(mask, value):
        m = mask & ~done
        out[m] = value[m] if isinstance(value, np.ndarray) else value
        done[m] = True

    neutral = pr == "NeutralPlay"
    put(neutral & isin(pc, FOUL_CALLS), "FoulBall")
    put(neutral & (pc == "StrikeCalled"), "StrikeCalled")
    put(neutral, None)
    put(pr == "Strikeout", pc)
    put(isin(pr, ["Error", "FieldersChoice", "Sacrifice"]), "Out")
    put(pr == "BallinDirt", "BallCalled")
    put(pr == "FoulBallFieldable", "FoulBall")

    ev = pd.Series(out, index=df.index, dtype="object")
    if rescue:
        resc = ev.isna() & df["PitchCall"].isin(list(RESCUE_MAP))
        print(f"Pitcheos rescatados desde PitchCall: {int(resc.sum()):,}")
        ev[resc] = df.loc[resc, "PitchCall"].map(RESCUE_MAP)
    df["event_type"] = ev
    return df


df = set_event_type(df)
print("\nEventos:")
print(df["event_type"].value_counts(dropna=False).to_string())
print("\nSin evento:", int(df["event_type"].isna().sum()))

In [ ]:
# Diagnóstico: qué PitchCall / play_result quedaron sin evento (para decidir si hace falta otra regla)
sin_evento = df[df["event_type"].isna()]
if len(sin_evento):
    display(sin_evento.groupby(["PitchCall", "play_result"], dropna=False).size()
            .sort_values(ascending=False).head(15).to_frame("n"))

## 4. Run values (`delta_run_exp` desde el CSV)

El CSV tiene el cambio promedio de carreras esperadas por **evento × conteo** (valor *context-neutral*: no depende de outs ni corredores).
Primero se revisa la tabla (orden lógico, valores repetidos, valores "redondos" que parezcan topes) y luego se une al dataframe.

In [ ]:
def load_run_values(path: str) -> pd.DataFrame:
    rv = pd.read_csv(path)
    need = {"event", "balls", "strikes", "delta_run_exp"}
    if not need.issubset(rv.columns):
        raise ValueError(f"El CSV debe tener columnas {need}")
    return rv


def validate_run_value_table(rv: pd.DataFrame) -> None:
    problems = []
    print("Eventos:", sorted(rv["event"].unique()))
    print("Llaves duplicadas (event,balls,strikes):", int(rv.duplicated(["event", "balls", "strikes"]).sum()))
    piv = rv.pivot_table(index=["balls", "strikes"], columns="event", values="delta_run_exp")

    order = ["Single", "Double", "Triple", "HomeRun"]          # 1) orden esperado de los hits
    for (b, s), row in piv.iterrows():
        for e1, e2 in zip(order[:-1], order[1:]):
            v1, v2 = row.get(e1, np.nan), row.get(e2, np.nan)
            if pd.notna(v1) and pd.notna(v2) and v2 < v1 - 1e-9:
                problems.append(f"{b}-{s}: {e2} ({v2:.3f}) < {e1} ({v1:.3f})  [orden]")

    for (b, s), row in piv.iterrows():                          # 2) mismo valor en eventos distintos
        r = row.dropna().round(6)
        for val, grp in r.groupby(r):
            ev = sorted(grp.index)
            legit = set(ev) == {"BallCalled", "Walk"} or (s == 2 and set(ev) == {"StrikeCalled", "StrikeSwinging"})
            if len(ev) > 1 and not legit:   # (bola/Walk y strike cantado/swinging con 2 strikes valen igual a propósito)
                problems.append(f"{b}-{s}: mismo valor {val:.3f} en {ev}  [posible relleno]")

    for _, r in rv.iterrows():                                  # 3) valores "redondos" (posible tope)
        v = r["delta_run_exp"]
        if abs(v * 100 - round(v * 100)) < 1e-9 and abs(v) > 0:
            problems.append(f"{r['event']} {int(r['balls'])}-{int(r['strikes'])}: valor redondo {v}  [posible tope]")

    print("\nAvisos:" if problems else "\nSin avisos.")
    for p in problems:
        print(" -", p)


def attach_delta_run_exp(df: pd.DataFrame, rv: pd.DataFrame) -> pd.DataFrame:
    key = rv.rename(columns={"event": "event_type", "balls": "Balls", "strikes": "Strikes"})
    out = df.merge(key, on=["event_type", "Balls", "Strikes"], how="left", validate="m:1")
    assert len(out) == len(df), "El merge cambió el número de filas"

    # Respaldo: bola con 3 bolas (el CSV la trae como Walk)
    walk = rv[rv["event"] == "Walk"].set_index(["balls", "strikes"])["delta_run_exp"]
    miss = out["delta_run_exp"].isna() & (out["event_type"] == "BallCalled") & (out["Balls"] == 3)
    for s in (0, 1, 2):
        m = miss & (out["Strikes"] == s)
        if m.any() and (3, s) in walk.index:
            out.loc[m, "delta_run_exp"] = walk[(3, s)]
    out["has_rv"] = out["delta_run_exp"].notna()
    return out


df_run_values = load_run_values(RUN_VALUES_PATH)
validate_run_value_table(df_run_values)

df = attach_delta_run_exp(df, df_run_values)
print(f"\nPitcheos sin run value: {int((~df['has_rv']).sum()):,} de {len(df):,}")
if (~df["has_rv"]).any():
    display(df.loc[~df["has_rv"]].groupby(["event_type", "Balls", "Strikes"], dropna=False)
            .size().sort_values(ascending=False).head(10).to_frame("n"))

## 5. Funciones de plot

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon

def plot_heat_map_strike_zone_by_column(df, column, pitcher_id=None, pitch_type=None, gridsize=25, min_cnt=2, invert_side=True,title=None
):
    """
    Grafica un mapa de calor en la zona de strike para cualquier columna.
    
    - Si la columna es binaria (0/1): Filtra los 1s y muestra densidad de ocurrencia.
    - Si la columna es continua/probabilidad: Muestra la probabilidad promedio en cada zona.
    
    Parámetros:
    -----------
    df : pd.DataFrame con 'PlateLocSide' y 'PlateLocHeight'.
    column : str, nombre de la columna a graficar (ej. 'is_whiff', 'pred_p_whiff', 'Stuff_Plus').
    pitcher_id : str (opcional), filtra a un lanzador específico 
    pitch_type : str (opcional), filtra a un tipo de pitcheo (ej. 'Slider', 'Four-Seam').
    gridsize : int, resolución de los hexágonos del heatmap (default: 25).
    min_cnt : int, mínimo de lanzamientos en un hexágono para mostrar color (default: 2).
    invert_side : bool, si True invierte el eje X para vista desde el catcher/umpire (default: True).
    """
    required = ['PlateLocSide', 'PlateLocHeight', column]
    for col in required:
        if col not in df.columns:
            raise ValueError(f"La columna '{col}' no existe en el DataFrame.")

    # 1. Filtros opcionales de pitcher y tipo de lanzamiento
    data = df.dropna(subset=['PlateLocSide', 'PlateLocHeight', column]).copy()
    filter_desc = []
    
    if pitcher_id is not None:
        data = data[data['pitcher_anon_id'] == pitcher_id]
        filter_desc.append(f"Pitcher: {pitcher_id}")
    if pitch_type is not None:
        data = data[data['AutoPitchType'] == pitch_type]
        filter_desc.append(f"Tipo: {pitch_type}")
        
    if len(data) == 0:
        print("No hay datos disponibles para los filtros seleccionados.")
        return

    # Ajuste de perspectiva (Catchers view: PlateLocSide * -1)
    x = data['PlateLocSide'] * (-1 if invert_side else 1)
    y = data['PlateLocHeight']
    vals = data[column]

    # 2. Detectar si es columna binaria o continua / probabilidad
    unique_vals = set(vals.dropna().unique())
    is_binary = unique_vals.issubset({0, 1, 0.0, 1.0, True, False}) and len(unique_vals) <= 2

    fig, ax = plt.subplots(figsize=(8, 9), facecolor='white')

    # =========================================================================
    # CASO A: Columna Binaria (ej. is_whiff, is_called_strike)
    # =========================================================================
    if is_binary:
        mask_pos = (vals == 1) | (vals == True)
        x_filtered = x[mask_pos]
        y_filtered = y[mask_pos]
        n_pos = len(x_filtered)
        pct = (n_pos / max(len(data), 1)) * 100
        
        hb = ax.hexbin(x_filtered, y_filtered, gridsize=gridsize, cmap='YlOrRd', mincnt=1,edgecolors='none',zorder=3)
        cbar = plt.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(f'Frecuencia de {column} == 1 (Conteo)', fontsize=11, fontweight='semibold')
        mode_label = f"Eventos: {n_pos:,} / {len(data):,} ({pct:.1f}%)"

    # =========================================================================
    # CASO B: Columna Continua / Probabilidad (ej. pred_p_whiff, Stuff_Plus)
    # =========================================================================
    else:
        hb = ax.hexbin(x, y, C=vals, reduce_C_function=np.mean, gridsize=gridsize, cmap='coolwarm', mincnt=min_cnt,edgecolors='none',zorder=3)
        cbar = plt.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)
        
        # Etiqueta de la barra según si es probabilidad o índice
        if vals.max() <= 1.0 and vals.min() >= 0.0:
            cbar.set_label(f'Probabilidad Promedio de {column}', fontsize=11, fontweight='semibold')
            cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f"{val*100:.1f}%"))
            mode_label = f"Prob. Global: {vals.mean()*100:.1f}% (n={len(data):,})"
        else:
            cbar.set_label(f'Valor Promedio de {column}', fontsize=11, fontweight='semibold')
            mode_label = f"Media: {vals.mean():.2f} (n={len(data):,})"

    # =========================================================================
    # 3. Dibujar Zona de Strike y Home Plate
    # =========================================================================
    zone_width = 1.66       # 17 pulgadas + radio bola en pies
    zone_left = -zone_width / 2
    top_of_zone = 3.5
    bottom_of_zone = 1.6
    zone_height = top_of_zone - bottom_of_zone

    # Rectángulo exterior de la zona
    strike_zone = Rectangle((zone_left, bottom_of_zone),zone_width,zone_height,facecolor='none',
        edgecolor='black',linewidth=2.5,zorder=5,label='Zona de Strike')
    ax.add_patch(strike_zone)

    # Cuadrícula interna 3x3
    w_third = zone_width / 3
    h_third = zone_height / 3
    for i in range(1, 3):
        # Líneas verticales
        ax.plot([zone_left + i * w_third, zone_left + i * w_third],
                [bottom_of_zone, top_of_zone], color='black', linestyle=':', linewidth=1.2, zorder=5)
        # Líneas horizontales
        ax.plot([zone_left, zone_left + zone_width],
                [bottom_of_zone + i * h_third, bottom_of_zone + i * h_third], color='black', linestyle=':', linewidth=1.2, zorder=5)

    # Pentágono del Home Plate (vista aérea colocada abajo como referencia)
    plate_y = 0.5
    plate_pts = np.array([
        [-0.708, plate_y],
        [0.708, plate_y],
        [0.708, plate_y - 0.2],
        [0, plate_y - 0.4],
        [-0.708, plate_y - 0.2]
    ])
    home_plate = Polygon(plate_pts, closed=True, facecolor='lightgray', edgecolor='black', linewidth=1.5, zorder=4)
    ax.add_patch(home_plate)

    # =========================================================================
    # 4. Estilo, Ejes y Títulos
    # =========================================================================
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6, zorder=2)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(0, 5.0)
    ax.set_aspect('equal')
    ax.set_xlabel('PlateLocSide (pies) - Perspectiva del Catcher', fontsize=11)
    ax.set_ylabel('PlateLocHeight (pies)', fontsize=11)
    ax.grid(alpha=0.15, linestyle='--')

    # Título principal y subtítulo
    filt_txt = f" [{', '.join(filter_desc)}]" if filter_desc else ""
    plot_title = title if title else f"Distribución de {column} en Zona de Strike{filt_txt}"
    ax.set_title(f"{plot_title}\n{mode_label}", fontsize=12, fontweight='bold', pad=12)

    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_heat_map_2d(df,x_col,y_col,target_col,pitcher_id=None,pitch_type=None,gridsize=30,min_cnt=3,cmap='coolwarm',clean_outliers=True,title=None):
    """
    Genera un mapa de calor 2D cruzando dos métricas físicas (x_col, y_col)
    y coloreando cada región según el valor promedio de target_col.
    
    Parámetros:
    -----------
    df : pd.DataFrame con los datos.
    x_col : str, nombre de la columna para el eje X (ej. 'SpinRate', 'Extension', 'HorzBreak').
    y_col : str, nombre de la columna para el eje Y (ej. 'InducedVertBreak', 'RelSpeed').
    target_col : str, columna objetivo para el color (ej. 'is_whiff', 'pred_p_whiff', 'Stuff_Plus').
    pitcher_id : str (opcional), filtra por ID de pitcher 
    pitch_type : str (opcional), filtra por tipo de pitcheo (ej. 'Four-Seam', 'Slider').
    gridsize : int, resolución de los hexágonos en el plano (default: 30).
    min_cnt : int, lanzamientos mínimos por hexágono para pintarlo (default: 3).
    cmap : str, paleta de colores ('coolwarm', 'viridis', 'plasma', 'YlOrRd', etc.).
    clean_outliers : bool, recorta percentiles extremos (1% y 99%) para evitar ruido de TrackMan.
    """
    cols_needed = [x_col, y_col, target_col]
    for c in cols_needed:
        if c not in df.columns:
            raise ValueError(f"La columna '{c}' no existe en el DataFrame.")

    # 1. Filtrar datos limpios y filtros opcionales
    data = df.dropna(subset=cols_needed).copy()
    filter_tags = []
    
    if pitcher_id is not None:
        data = data[data['pitcher_anon_id'] == pitcher_id]
        filter_tags.append(f"Pitcher: {pitcher_id}")
    if pitch_type is not None:
        data = data[data['AutoPitchType'] == pitch_type]
        filter_tags.append(f"Tipo: {pitch_type}")
        
    if len(data) == 0:
        print("No hay suficientes datos para los filtros seleccionados.")
        return

    # Limpieza de valores atípicos extremos para que los ejes queden bien encuadrados
    if clean_outliers:
        q_low_x, q_hi_x = data[x_col].quantile(0.005), data[x_col].quantile(0.995)
        q_low_y, q_hi_y = data[y_col].quantile(0.005), data[y_col].quantile(0.995)
        data = data[(data[x_col] >= q_low_x) & (data[x_col] <= q_hi_x) &
                    (data[y_col] >= q_low_y) & (data[y_col] <= q_hi_y)]

    x = data[x_col].values
    y = data[y_col].values
    vals = data[target_col].values

    # 2. Detectar si el target es binario (0/1) o continuo
    unique_vals = set(pd.Series(vals).dropna().unique())
    is_binary = unique_vals.issubset({0, 1, 0.0, 1.0, True, False}) and len(unique_vals) <= 2

    fig, ax = plt.subplots(figsize=(10, 7.5), facecolor='white')

    # 3. Graficar Hexbin 2D con agregación por promedio (np.mean)
    hb = ax.hexbin(x, y, C=vals.astype(float), reduce_C_function=np.mean, gridsize=gridsize, cmap=cmap, mincnt=min_cnt,edgecolors='gray',linewidths=0.2,zorder=3)

    # 4. Barra de color y leyenda según el tipo de target
    cbar = plt.colorbar(hb, ax=ax, fraction=0.046, pad=0.03)
    
    if is_binary:
        cbar.set_label(f'Tasa Promedio de {target_col} (% de éxito)', fontsize=11, fontweight='semibold')
        cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f"{val*100:.1f}%"))
        stat_text = f"Tasa Global {target_col}: {vals.mean()*100:.1f}%\nMuestra: {len(data):,} pitcheos"
    elif vals.max() <= 1.0 and vals.min() >= 0.0:
        cbar.set_label(f'Probabilidad Promedio de {target_col}', fontsize=11, fontweight='semibold')
        cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f"{val*100:.1f}%"))
        stat_text = f"Prob. Promedio: {vals.mean()*100:.1f}%\nMuestra: {len(data):,} pitcheos"
    else:
        cbar.set_label(f'Valor Promedio de {target_col}', fontsize=11, fontweight='semibold')
        stat_text = f"Media {target_col}: {vals.mean():.2f}\nMuestra: {len(data):,} pitcheos"

    # 5. Líneas de referencia (medias de X y Y)
    ax.axvline(x.mean(), color='black', linestyle=':', linewidth=1.2, alpha=0.6, label=f'Media {x_col}: {x.mean():.1f}')
    ax.axhline(y.mean(), color='black', linestyle=':', linewidth=1.2, alpha=0.6, label=f'Media {y_col}: {y.mean():.1f}')
    
    # Si alguna métrica es quiebre horizontal/vertical, marcar el 0 de quiebre neutro
    if 'break' in x_col.lower():
        ax.axvline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
    if 'break' in y_col.lower():
        ax.axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.5)

    # 6. Estética y títulos
    ax.set_xlabel(f"{x_col}", fontsize=11, fontweight='semibold')
    ax.set_ylabel(f"{y_col}", fontsize=11, fontweight='semibold')
    ax.grid(alpha=0.25, linestyle='--')
    ax.legend(loc='upper right', framealpha=0.9, fontsize=9)

    tag_str = f" [{', '.join(filter_tags)}]" if filter_tags else ""
    plot_title = title if title else f"Mapa de Calor 2D: {x_col} vs {y_col}\n(Color = {target_col}){tag_str}"
    ax.set_title(plot_title, fontsize=13, fontweight='bold', pad=12)

    # Caja informativa en la esquina inferior izquierda
    ax.text(
        0.03, 0.04, stat_text,
        transform=ax.transAxes,
        fontsize=9,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='lightgray'),
        zorder=4
    )

    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon


def plot_pitches(dataframe, batter_height=None, color_by='AutoPitchType'):
    if batter_height is not None:
        if 10 < batter_height < 300:
            batter_height_feet = batter_height / 30.48
        else:
            batter_height_feet = batter_height
        top_of_zone = batter_height_feet * 0.51
        bottom_of_zone = batter_height_feet * 0.22
    else:
        top_of_zone = 3.5
        bottom_of_zone = 1.6

    zone_width = 1.66
    zone_left = -zone_width / 2
    required_columns = ['PlateLocSide', 'PlateLocHeight']
    if color_by in dataframe.columns:
        required_columns.append(color_by)

    pitch_data = dataframe[required_columns].dropna()
    fig, ax = plt.subplots(figsize=(10, 8))

    if color_by in pitch_data.columns:
        pitch_types = sorted(pitch_data[color_by].unique())
        colors = plt.cm.tab10(range(len(pitch_types)))
        color_map = dict(zip(pitch_types, colors))
        for pitch_type in pitch_types:
            group = pitch_data[pitch_data[color_by] == pitch_type]
            ax.scatter(
                group['PlateLocSide'] * -1,
                group['PlateLocHeight'],
                s=55,
                alpha=0.75,
                color=color_map[pitch_type],
                edgecolors='black',
                linewidths=0.3,
                label=f'{pitch_type} (n={len(group):,})',
                zorder=4
            )
    else:
        ax.scatter(
            pitch_data['PlateLocSide'] * -1,
            pitch_data['PlateLocHeight'],
            s=55,
            alpha=0.75,
            color='steelblue',
            edgecolors='black',
            linewidths=0.3,
            label=f'Pitches (n={len(pitch_data):,})',
            zorder=4
        )

    strike_zone = Rectangle(
        (zone_left, bottom_of_zone),
        zone_width,
        top_of_zone - bottom_of_zone,
        facecolor='tomato',
        edgecolor='black',
        linewidth=2,
        alpha=0.20,
        label='Zona de strike',
        zorder=2
    )
    ax.add_patch(strike_zone)

    third_w = zone_width / 3
    third_h = (top_of_zone - bottom_of_zone) / 3
    for i in range(1, 3):
        ax.plot(
            [zone_left + i * third_w, zone_left + i * third_w],
            [bottom_of_zone, top_of_zone],
            color='black', linewidth=1, alpha=0.3, zorder=3
        )
        ax.plot(
            [zone_left, zone_left + zone_width],
            [bottom_of_zone + i * third_h, bottom_of_zone + i * third_h],
            color='black', linewidth=1, alpha=0.3, zorder=3
        )

    ax.axvline(0, color='gray', linewidth=1, linestyle='--', alpha=0.7)
    title_suffix = f' (Altura bateador: {batter_height})' if batter_height else ''
    ax.set_title(
        f'Ubicación de los pitches por tipo y zona de strike{title_suffix}',
        fontsize=13,
        fontweight='bold'
    )
    ax.set_xlabel('PlateLocSide (pies)')
    ax.set_ylabel('PlateLocHeight (pies)')
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(0, 5)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(alpha=0.25)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1))
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_stuff_vs_location_scatter(df_pitchers, min_pitches=100):
  """Grafica el cuadrante de desempeño: Stuff+ (eje X) vs Location+ (eje Y)."""
  agg = (
      df_pitchers.groupby('pitcher_anon_id')
      .agg(
          n_pitches=('Location_Plus', 'count'),
          stuff_plus=('Stuff_Plus_PitchType', 'mean'),
          location_plus=('Location_Plus', 'mean'),
      )
      .query('n_pitches >= @min_pitches')
  )

  fig, ax = plt.subplots(figsize=(9, 8))
  scatter = ax.scatter(
      agg['stuff_plus'],
      agg['location_plus'],
      c=agg['n_pitches'],
      cmap='Blues',
      alpha=0.75,
      edgecolors='black',
      s=60,
  )

  # Líneas de media de liga (100)
  ax.axvline(100, color='red', linestyle='--', linewidth=1.2, alpha=0.7)
  ax.axhline(100, color='red', linestyle='--', linewidth=1.2, alpha=0.7)

  # Etiquetas de cuadrantes
  ax.text(
      112,
      110,
      'AS / ELITE\n(Stuff Sucio + Gran Comando)',
      fontsize=10,
      fontweight='bold',
      color='darkgreen',
      ha='center',
  )
  ax.text(
      88,
      110,
      'CONTROL / FINESSE\n(Poco Stuff + Alto Comando)',
      fontsize=10,
      fontweight='bold',
      color='navy',
      ha='center',
  )
  ax.text(
      112,
      90,
      'TIRADORES / RAW\n(Gran Stuff + Descontrol)',
      fontsize=10,
      fontweight='bold',
      color='darkorange',
      ha='center',
  )
  ax.text(
      88,
      90,
      'VULNERABLES\n(Bajo Stuff + Bajo Comando)',
      fontsize=10,
      fontweight='bold',
      color='darkred',
      ha='center',
  )

  ax.set_xlabel('Stuff+ Promedio', fontsize=12, fontweight='semibold')
  ax.set_ylabel('Location+ Promedio', fontsize=12, fontweight='semibold')
  ax.set_title(
      f'Perfil de Lanzadores: Stuff+ vs Location+ (Mínimo {min_pitches} pitcheos)',
      fontsize=13,
      fontweight='bold',
  )
  cbar = plt.colorbar(scatter, ax=ax)
  cbar.set_label('Total de Pitcheos', fontsize=10)
  ax.grid(alpha=0.3, linestyle='--')
  plt.tight_layout()
  plt.show()

## 6. Targets auxiliares (como los tenías)

Son los targets binarios que ya construías (whiff, chase, strike cantado, contacto débil, rolling, hard contact, out en la aparición).
Ya **no entran al xRV**: sirven para diagnosticar y para comparar contra lo que predicen los modelos de las secciones 9 y 10.

In [ ]:
def build_whiff(df):
    df = df.copy()
    if "is_whiff" not in df.columns:
        df["is_whiff"] = (df["event_type"] == "StrikeSwinging")
    print("is_whiff = 1:", int(df["is_whiff"].sum()))
    return df


def build_chase(df):
    df = df.copy()
    if "is_swing" not in df.columns:
        df["is_swing"] = df["event_type"].isin(["StrikeSwinging", "FoulBall", "Out", "Single", "Double", "Triple", "HomeRun"])
    if "outside_strike_zone" not in df.columns:
        df["outside_strike_zone"] = (df["in_strike_zone"] == 0).astype(int)
    if "swung_outside_strike_zone" not in df.columns:
        df["swung_outside_strike_zone"] = ((df["outside_strike_zone"] == 1) & df["is_swing"].astype(bool)).astype(int)
    print("swung_outside_strike_zone = 1:", int(df["swung_outside_strike_zone"].sum()))
    return df


def build_called_strike(df):
    df = df.copy()
    if "is_called_strike" not in df.columns:
        df["is_called_strike"] = (df["event_type"] == "StrikeCalled")
    print("is_called_strike = 1:", int(df["is_called_strike"].sum()))
    return df


def build_weak_contact(df, threshold_mph=71):
    df = df.copy()
    es = pd.to_numeric(df["ExitSpeed"], errors="coerce") if "ExitSpeed" in df.columns else pd.Series(np.nan, index=df.index)
    df["is_weak_contact"] = (es < threshold_mph).fillna(False).astype(float)
    print("is_weak_contact = 1:", int(df["is_weak_contact"].sum()))
    return df


def build_groundball(df):
    df = df.copy()
    ht = df["hit_type"] if "hit_type" in df.columns else pd.Series(None, index=df.index, dtype="object")
    df["is_groundball"] = (ht == "GroundBall").astype(float)
    print("is_groundball = 1:", int(df["is_groundball"].sum()))
    return df


def build_barrel_hard_contact(df, quantile=0.85):
    """1 si fue LineDrive/FlyBall con ExitSpeed >= percentil 85 de la propia liga (no un umbral fijo de MLB)."""
    df = df.copy()
    if "hit_type" not in df.columns or "ExitSpeed" not in df.columns:
        df["barrel_hard_contact"] = 0.0
        return df
    es = pd.to_numeric(df["ExitSpeed"], errors="coerce")
    batted = df["hit_type"].notna() & (df["hit_type"] != "None")
    thr = es[batted].quantile(quantile)
    print(f"Umbral hard contact (P{int(quantile*100)} en batted balls): {thr:.1f} mph")
    df["barrel_hard_contact"] = (df["hit_type"].isin(["LineDrive", "FlyBall"]) & (es >= thr)).astype(float)
    print("barrel_hard_contact = 1:", int(df["barrel_hard_contact"].sum()))
    return df


def build_out_within_plate_appearance(df):
    """1 si el resultado de la aparición fue un out del bateador (ponche u out en juego)."""
    df = df.copy()
    df["out_within_pa"] = df["event_type"].isin(["Out"]).astype(float)
    # ponches: StrikeSwinging/StrikeCalled con play_result == Strikeout
    df.loc[df["play_result"] == "Strikeout", "out_within_pa"] = 1.0
    print("out_within_pa = 1:", int(df["out_within_pa"].sum()))
    return df


def build_auxiliary_targets(df):
    df = build_whiff(df)
    df = build_chase(df)
    df = build_called_strike(df)
    df = build_weak_contact(df)
    df = build_groundball(df)
    df = build_barrel_hard_contact(df)
    df = build_out_within_plate_appearance(df)
    return df


df = build_auxiliary_targets(df)

## 7. xRunValue: clases y tabla de valores

**Idea:** un modelo multiclase predice la probabilidad de cada resultado y el xRV es `Σ P(clase) × RunValue(clase)`.

| | **Stuff+** | **Location+** |
|---|---|---|
| Filas de entrenamiento | solo **swings** | **todos** los pitcheos |
| Clases | StrikeSwinging, FoulBall, Out, Single, XBH, HomeRun | Ball, StrikeCalled, StrikeSwinging, FoulBall, Out, Single, XBH, HomeRun |
| Features | física del pitcheo (sin ubicación) | ubicación + conteo + lateralidad |
| Valor de cada clase | promedio neutral (o por conteo) | promedio **por conteo** |

- `XBH` junta doble y triple (el triple es raro y en tu CSV tiene valores inestables).
- `Ball` junta bola, base por bolas y HBP.
- El valor de cada clase sale de **tus datos**: el promedio de `delta_run_exp` de las filas de esa clase. Así pesa por frecuencia real y no depende de las celdas raras del CSV.

In [ ]:
STUFF_CLASSES = ["StrikeSwinging", "FoulBall", "Out", "Single", "XBH", "HomeRun"]
LOC_CLASSES   = ["Ball", "StrikeCalled", "StrikeSwinging", "FoulBall", "Out", "Single", "XBH", "HomeRun"]

_SWING_MAP = {"StrikeSwinging": "StrikeSwinging", "FoulBall": "FoulBall", "Out": "Out", "Single": "Single",
              "Double": "XBH", "Triple": "XBH", "HomeRun": "HomeRun"}
STUFF_MAP = dict(_SWING_MAP)
LOC_MAP = {**_SWING_MAP, "BallCalled": "Ball", "Walk": "Ball", "HitByPitch": "Ball", "StrikeCalled": "StrikeCalled"}


def add_class_targets(df):
    df = df.copy()
    df["stuff_class"] = df["event_type"].map(STUFF_MAP)   # NaN si no hubo swing
    df["loc_class"]   = df["event_type"].map(LOC_MAP)
    return df


def build_rv_tables(df):
    d = df[df["delta_run_exp"].notna()]
    sw = d[d["stuff_class"].notna()]
    lc = d[d["loc_class"].notna()]
    return {
        "stuff_neutral": sw.groupby("stuff_class")["delta_run_exp"].mean().reindex(STUFF_CLASSES),
        "stuff_count":   sw.groupby(["stuff_class", "Balls", "Strikes"])["delta_run_exp"].mean(),
        "loc_neutral":   lc.groupby("loc_class")["delta_run_exp"].mean().reindex(LOC_CLASSES),
        "loc_count":     lc.groupby(["loc_class", "Balls", "Strikes"])["delta_run_exp"].mean(),
    }


def rv_array(rv_count, rv_neutral, classes):
    """Arreglo (K, 4 bolas, 3 strikes) con el valor de cada clase por conteo (respaldo: valor neutral)."""
    neutral = rv_neutral.reindex(classes).to_numpy(dtype=float)
    arr = np.tile(neutral[:, None, None], (1, 4, 3))
    for (k, b, s), v in rv_count.items():
        if k in classes and 0 <= b <= 3 and 0 <= s <= 2:
            arr[classes.index(k), int(b), int(s)] = v
    return arr


def rv_matrix(balls, strikes, arr, neutral_vec, mode):
    """Matriz (n, K) con el valor de cada clase para cada pitcheo."""
    n = len(balls)
    if mode == "neutral":
        return np.tile(neutral_vec, (n, 1))
    b = pd.to_numeric(balls, errors="coerce")
    s = pd.to_numeric(strikes, errors="coerce")
    bad = (b.isna() | s.isna() | (b < 0) | (b > 3) | (s < 0) | (s > 2)).to_numpy()
    bi = b.fillna(0).clip(0, 3).astype(int).to_numpy()
    si = s.fillna(0).clip(0, 2).astype(int).to_numpy()
    M = arr[:, bi, si].T.copy()
    M[bad] = neutral_vec
    return M


df = add_class_targets(df)
rv_tables = build_rv_tables(df)

print("Valor neutral por clase (Stuff, solo swings):")
display(rv_tables["stuff_neutral"].round(3).to_frame("run_value"))
print("Valor por conteo (Stuff):")
display(rv_tables["stuff_count"].unstack("stuff_class").reindex(columns=STUFF_CLASSES).round(3))
print("Valor neutral por clase (Location):")
display(rv_tables["loc_neutral"].round(3).to_frame("run_value"))

##  limpiar outliers

In [ ]:
"""
DIAGNÓSTICO DE OUTLIERS — features de Stuff+ y Location+
=========================================================
Uso en el notebook (después de la sección 8, antes de entrenar):

    exec(open("diagnostico_outliers.py", encoding="utf-8").read())   # o pega el contenido en una celda
    flags, report = run_outlier_diagnostics(df, STUFF_FEATURES, LOC_FEATURES)
    df = df.join(flags)                                              # agrega columnas out_*

Después de entrenar (sección 9), para explicar los Stuff+ extremos:

    explain_extreme_scores(df, "Stuff_Plus_PitchType", STUFF_FEATURES)

Y para decidir qué hacer con lo encontrado (en este orden):

    df_clean, log = apply_outlier_policy(df, STUFF_FEATURES + LOC_FEATURES, policy="nan_hard")  # imposibles físicos
    df_clean, log2 = nan_shifted_features(df_clean, flags)   # feature corrida en (juego, tipo) / HorzBreak invertido

Los límites físicos (HARD_LIMITS) asumen unidades Trackman: mph, rpm, pulgadas, pies y grados.
Ajústalos si tus datos usan otras unidades.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from IPython.display import display

# ------------------------------- Límites físicos plausibles -----------------------------
HARD_LIMITS = {
    "RelSpeed": (50, 106), "SpinRate": (300, 3600), "SpinAxis": (0, 360),
    "InducedVertBreak": (-30, 30), "HorzBreak": (-35, 35), "HorzBreak_m": (-35, 35),
    "RelHeight": (2.5, 8.0), "RelSide": (-5.5, 5.5), "RelSide_m": (-5.5, 5.5),
    "Extension": (3.5, 8.5),
    "VertRelAngle": (-8, 5), "HorzRelAngle": (-8, 8), "HorzRelAngle_m": (-8, 8),
    "PlateLocHeight": (-1.5, 6.5), "PlateLocSide": (-4.0, 4.0),
    "speed_diff": (-45, 12), "ivb_diff": (-55, 45), "hb_diff": (-65, 65),
    "density_index": (0.6, 1.2),
    "Balls": (0, 3), "Strikes": (0, 2),
}
# Features que definen si un pitcheo se puede usar (si fallan, el pitcheo no sirve)
CIRCULAR = ["SpinAxis_sin", "SpinAxis_cos"]   # el z robusto no aplica a variables circulares (se quedan en el multivariado)
GAME_LEVEL = ["density_index"]          # features que son constantes por juego: se revisan a nivel juego, no pitcheo
CORE_STUFF = ["RelSpeed", "InducedVertBreak", "HorzBreak_m"]
CORE_LOC = ["PlateLocHeight", "PlateLocSide"]


# ================================== 1. INTEGRIDAD BÁSICA ==================================
def check_basic_integrity(df, pitch_uid_col="PitchUID"):
    print("=" * 70, "\n1) INTEGRIDAD BÁSICA\n", "=" * 70, sep="")
    out = {}
    if pitch_uid_col in df.columns:
        out["PitchUID duplicados"] = int(df[pitch_uid_col].duplicated().sum())
    num = df.select_dtypes(include=[np.number])
    out["valores infinitos"] = int(np.isinf(num.to_numpy(dtype=float, na_value=np.nan)).sum())
    if {"Balls", "Strikes"}.issubset(df.columns):
        bad = ~(df["Balls"].between(0, 3) & df["Strikes"].between(0, 2))
        out["conteos inválidos (Balls∉0-3 o Strikes∉0-2, incl. NaN)"] = int(bad.sum())
    for k, v in out.items():
        print(f"  {k}: {v:,}" + ("   <-- revisar" if v else ""))
    return out


# ================================== 2. LÍMITES FÍSICOS ==================================
def hard_limit_report(df, cols, limits=HARD_LIMITS):
    """Cuenta valores faltantes y fuera de rango físico por feature. Devuelve (tabla, máscara de violaciones)."""
    rows, viol = [], pd.DataFrame(False, index=df.index, columns=[c for c in cols if c in df.columns])
    for c in cols:
        if c not in df.columns:
            continue
        x = pd.to_numeric(df[c], errors="coerce")
        lo, hi = limits.get(c, (-np.inf, np.inf))
        below, above = (x < lo), (x > hi)
        viol[c] = (below | above).fillna(False)
        rows.append({"feature": c, "pct_NaN": 100 * x.isna().mean(), "n_bajo_min": int(below.sum()),
                     "n_sobre_max": int(above.sum()), "min_obs": x.min(), "max_obs": x.max(),
                     "limite": f"[{lo}, {hi}]"})
    return pd.DataFrame(rows), viol


# ================================== 3. OUTLIERS ROBUSTOS ==================================
def robust_z(df, cols, by):
    """z robusto (mediana y MAD) de cada feature dentro de cada grupo (tipo de pitcheo)."""
    Z = pd.DataFrame(index=df.index, columns=[c for c in cols if c in df.columns], dtype=float)
    for c in Z.columns:
        x = pd.to_numeric(df[c], errors="coerce")
        med = x.groupby(df[by]).transform("median")
        mad = (x - med).abs().groupby(df[by]).transform("median") * 1.4826
        sd = x.groupby(df[by]).transform("std")
        scale = mad.where(mad > 1e-9, sd).replace(0, np.nan)
        Z[c] = (x - med) / scale
    return Z


# ================================== 4. MULTIVARIADO ==================================
def multivariate_flags(df, cols, by, contamination=0.002, max_fit=80_000, seed=42):
    """IsolationForest por tipo de pitcheo: detecta COMBINACIONES raras aunque cada valor sea normal."""
    cols = [c for c in cols if c in df.columns and df[c].dtype != bool]
    flag = pd.Series(False, index=df.index)
    score = pd.Series(np.nan, index=df.index)
    for pt, g in df.groupby(by):
        X = g[cols].apply(pd.to_numeric, errors="coerce")
        ok = X.notna().all(axis=1)
        if ok.sum() < 500:
            continue
        Xo = X[ok]
        fit = Xo.sample(min(len(Xo), max_fit), random_state=seed)
        iso = IsolationForest(n_estimators=100, contamination=contamination, random_state=seed, n_jobs=-1).fit(fit)
        s = pd.Series(iso.score_samples(Xo), index=Xo.index)
        thr = np.quantile(iso.score_samples(fit), contamination)
        score.loc[Xo.index] = s
        flag.loc[Xo.index] = s < thr
    return flag, score


# ================================== 5. CONSISTENCIA FÍSICA ==================================
def consistency_flags(df, tol=0.06):
    """La rapidez del vector de velocidad inicial debe ser ≈ RelSpeed (mph × 1.4667 = ft/s)."""
    need = ["vx0", "vy0", "vz0", "RelSpeed"]
    if not all(c in df.columns for c in need):
        return pd.Series(False, index=df.index), pd.Series(np.nan, index=df.index)
    v = np.sqrt(df["vx0"] ** 2 + df["vy0"] ** 2 + df["vz0"] ** 2)
    ratio = v / (df["RelSpeed"] * 1.4667)
    return ((ratio - 1).abs() > tol).fillna(False), ratio


def game_density_flags(df, game_col="game_anon_id", altitude_col="altitude_category", z_thr=4.0, min_games=8):
    """Compara el density_index de cada juego contra los demás juegos de su misma categoría de altitud."""
    if "density_index" not in df.columns or game_col not in df.columns:
        return pd.Series(False, index=df.index), pd.DataFrame()
    g = df.groupby(game_col).agg(density=("density_index", "first"), alt=(altitude_col, "first"), n=("density_index", "size"))
    med = g.groupby("alt")["density"].transform("median")
    mad = (g["density"] - med).abs().groupby(g["alt"]).transform("median") * 1.4826
    sd = g.groupby("alt")["density"].transform("std")
    scale = mad.where(mad > 1e-4, sd).replace(0, np.nan)
    g["z"] = (g["density"] - med) / scale
    cnt = g.groupby("alt")["density"].transform("size")
    g["flag"] = (g["z"].abs() > z_thr) & (cnt >= min_games)
    bad = set(g.index[g["flag"]])
    return df[game_col].isin(bad), g[g["flag"]].sort_values("z", key=np.abs, ascending=False)


def game_shift_flags(df, game_col="game_anon_id", altitude_col="altitude_category", pitch_type_col="AutoPitchType",
                     cols=("RelSpeed", "InducedVertBreak", "HorzBreak_m", "SpinRate"),
                     z_thr=5.0, min_pitches=15, min_games=8, valid_mask=None):
    """Detecta juegos donde la MEDIANA de un tipo de pitcheo se corre mucho respecto a otros juegos de su misma
    altitud (típico de un radar/Trackman mal calibrado). El z incluye el error de muestreo esperado del juego
    (sd del pitcheo / raíz de n), para no marcar diferencias pequeñas en juegos con pocos pitcheos.
    valid_mask: filas a usar para calcular (ej. las que pasaron los límites físicos)."""
    cols = [c for c in cols if c in df.columns]
    keys = [game_col, pitch_type_col, altitude_col]
    if not cols or any(k not in df.columns for k in keys):
        return pd.Series(False, index=df.index), pd.DataFrame()
    d = df if valid_mask is None else df[valid_mask]
    lvl = [pitch_type_col, altitude_col]
    grp = d.groupby(keys, observed=True)
    g = grp[cols].median()
    n = grp.size()
    ok = n >= min_pitches
    g, n = g[ok], n[ok]
    med = g.groupby(level=lvl).transform("median")
    mad_between = (g - med).abs().groupby(level=lvl).transform("median") * 1.4826
    pitch_sd = d.groupby(lvl, observed=True)[cols].agg(lambda x: 1.4826 * (x - x.median()).abs().median())
    pitch_sd = pitch_sd.reindex(g.index.droplevel(0)).set_axis(g.index)
    se = 1.25 * pitch_sd.div(np.sqrt(n), axis=0)
    scale = np.sqrt(mad_between.fillna(0) ** 2 + se ** 2).replace(0, np.nan)
    n_games = g.groupby(level=lvl)[cols[0]].transform("size")
    Z = ((g - med) / scale).where(n_games >= min_games, np.nan)
    worst = Z.abs().fillna(-1).idxmax(axis=1)
    zmax = Z.abs().max(axis=1)
    tab = pd.DataFrame({"feature": worst,
                        "z": [Z.loc[i, w] if pd.notna(w) else np.nan for i, w in zip(Z.index, worst)],
                        "mediana_juego": [g.loc[i, w] for i, w in zip(g.index, worst)],
                        "mediana_otros": [med.loc[i, w] for i, w in zip(g.index, worst)],
                        "n_pitcheos": n.reindex(g.index).to_numpy()})
    tab = tab[(zmax > z_thr).to_numpy()].sort_values("z", key=np.abs, ascending=False)
    bad_pairs = [(i[0], i[1]) for i in tab.index]
    row_idx = pd.MultiIndex.from_arrays([df[game_col], df[pitch_type_col]])
    return pd.Series(row_idx.isin(bad_pairs), index=df.index), tab


def check_pileups(df, cols=("PlateLocHeight", "PlateLocSide", "RelSpeed", "SpinRate", "Extension", "RelHeight",
                            "InducedVertBreak", "HorzBreak"), min_share=0.0005):
    """Valores 'pegados' a un tope o a cero: señal de recorte (clipping) o de ceros usados como faltante."""
    rows = []
    for c in cols:
        if c not in df.columns:
            continue
        x = pd.to_numeric(df[c], errors="coerce").dropna()
        if x.empty:
            continue
        lo, hi = x.min(), x.max()
        vc = x.round(4).value_counts(normalize=True)
        rows.append({"feature": c, "min": lo, "% en el min": 100 * (x <= lo + 1e-9).mean(),
                     "max": hi, "% en el max": 100 * (x >= hi - 1e-9).mean(),
                     "n_exactamente_0": int((x == 0).sum()), "% 0": 100 * (x == 0).mean(),
                     "valor_mas_repetido": vc.index[0], "% repetido": 100 * vc.iloc[0]})
    t = pd.DataFrame(rows)
    t["sospechoso"] = ((t["% en el min"] > 100 * min_share) | (t["% en el max"] > 100 * min_share)
                       | (t["% 0"] > 100 * min_share) | (t["% repetido"] > 0.3))   # 0.3% de filas con EL MISMO valor continuo
    return t


def summarize_shift_table(flags):
    """Cuenta los juegos x tipo corridos por feature / tipo de pitcheo / altitud (para ver el patrón, no solo el top)."""
    st = flags.attrs.get("shift_table")
    if st is None or not len(st):
        print("Sin juegos corridos."); return None
    t = st.reset_index()
    t["signo"] = np.where(t["z"] > 0, "arriba", "abajo")
    cols = [c for c in t.columns if c in ("AutoPitchType", "altitude_category", "feature", "signo")]
    out = t.groupby(cols).agg(n_juegos=("z", "size"), z_mediano=("z", "median"),
                              mediana_juego=("mediana_juego", "median"), mediana_otros=("mediana_otros", "median"),
                              pitcheos=("n_pitcheos", "sum")).sort_values("n_juegos", ascending=False)
    return out


def hand_sign_flags(df, game_col="game_anon_id", group_col="pitcher_anon_id", pitch_type_col="AutoPitchType",
                    min_game_n=8, min_total_n=30, min_abs=3.0, min_share=0.5, min_pairs=3):
    """HorzBreak (SIN espejo) de un pitcher cambia de signo en un juego vs sus otros juegos: indica mano mal
    etiquetada o radar con orientación invertida. Marca juegos donde esto pasa en muchos pitchers a la vez."""
    need = ["HorzBreak", game_col, group_col, pitch_type_col]
    if any(c not in df.columns for c in need):
        return pd.Series(False, index=df.index), pd.DataFrame()
    k = [group_col, pitch_type_col]
    tot = df.groupby(k, observed=True)["HorzBreak"].agg(med_total="median", n_total="size")
    gm = df.groupby(k + [game_col], observed=True)["HorzBreak"].agg(med_game="median", n_game="size").reset_index()
    gm = gm.merge(tot.reset_index(), on=k, how="left")
    gm = gm[(gm["n_game"] >= min_game_n) & (gm["n_total"] >= min_total_n)
            & (gm["med_total"].abs() >= min_abs) & (gm["med_game"].abs() >= min_abs)]
    gm["invertido"] = np.sign(gm["med_game"]) != np.sign(gm["med_total"])
    byg = gm.groupby(game_col).agg(pares_evaluados=("invertido", "size"), pares_invertidos=("invertido", "sum"))
    byg["share"] = byg["pares_invertidos"] / byg["pares_evaluados"]
    bad = byg[(byg["share"] >= min_share) & (byg["pares_invertidos"] >= min_pairs)].sort_values("share", ascending=False)
    return df[game_col].isin(bad.index), bad


# ================================== 6. FLAGS ==================================
def flag_outliers(df, stuff_features, loc_features, pitch_type_col="AutoPitchType",
                  z_thr=6.0, contamination=0.002, game_col="game_anon_id", altitude_col="altitude_category",
                  game_escalate_rate=None, group_col="pitcher_anon_id"):
    cols = [c for c in dict.fromkeys(list(stuff_features) + list(loc_features)) if c in df.columns]
    num_cols = [c for c in cols if df[c].dtype != bool and c not in ("same_hand", "is_lhb", "is_lhp", "Balls", "Strikes") + tuple(GAME_LEVEL)]

    hard_tab, viol = hard_limit_report(df, cols)
    z_cols = [c for c in num_cols if c not in CIRCULAR]
    Z = robust_z(df, z_cols, pitch_type_col)
    f_multi, s_multi = multivariate_flags(df, [c for c in num_cols if c in stuff_features], pitch_type_col, contamination)
    f_cons, ratio = consistency_flags(df)
    f_dens, bad_games = game_density_flags(df, game_col, altitude_col)
    f_shift, shift_tab = game_shift_flags(df, game_col, altitude_col, pitch_type_col, valid_mask=~viol.any(axis=1))
    f_sign, sign_tab = hand_sign_flags(df, game_col, group_col, pitch_type_col)
    f_game = f_dens | f_shift | f_sign

    flags = pd.DataFrame(index=df.index)
    flags["out_hard"] = viol.any(axis=1)
    flags["out_robust"] = (Z.abs() > z_thr).any(axis=1)
    flags["out_multi"] = f_multi
    flags["out_consist"] = f_cons
    flags["out_game"] = f_game
    # Escalación OPCIONAL: si una fracción alta de los pitcheos de un juego ya está marcada, todo el juego es sospechoso
    base_flag = flags[["out_hard", "out_robust", "out_consist", "out_game"]].any(axis=1)
    rate = base_flag.groupby(df[game_col]).transform("mean")
    size = base_flag.groupby(df[game_col]).transform("size")
    if game_escalate_rate:   # opcional (apagado por defecto): marca juegos completos; castiga tipos que están bien
        flags["out_game"] = flags["out_game"] | ((rate >= game_escalate_rate) & (size >= 30))
    flags["out_n_flags"] = flags[["out_hard", "out_robust", "out_multi", "out_consist", "out_game"]].sum(axis=1)
    flags["out_any"] = flags["out_n_flags"] > 0
    flags["out_core_bad"] = viol[[c for c in CORE_STUFF + CORE_LOC if c in viol.columns]].any(axis=1) \
        | df[[c for c in CORE_STUFF + CORE_LOC if c in df.columns]].isna().any(axis=1)
    flags["iso_score"] = s_multi
    flags["vel_ratio"] = ratio

    # razón principal en las filas marcadas (solo ellas, para no gastar memoria)
    idx = flags.index[flags["out_any"]]
    reasons = pd.Series("", index=flags.index)
    if len(idx):
        zz = Z.loc[idx].abs()
        worst = zz.fillna(-1).idxmax(axis=1).where(zz.max(axis=1) > z_thr, "")
        hard_cols = viol.loc[idx].apply(lambda r: ",".join(r.index[r.to_numpy()]), axis=1)
        reasons.loc[idx] = ("HARD[" + hard_cols + "] ROBUST[" + worst.astype(str) + "]")
    flags["out_reasons"] = reasons
    flags.attrs["bad_games"] = bad_games
    flags.attrs["shift_table"] = shift_tab
    flags.attrs["sign_table"] = sign_tab
    return flags, hard_tab, Z


# ================================== 7. RESÚMENES ==================================
def summarize_flags(df, flags, by_cols, min_n=1):
    fl = [c for c in ["out_hard", "out_robust", "out_multi", "out_consist", "out_game", "out_any"] if c in flags.columns]
    keys = [df[c] for c in by_cols]
    pct = (flags[fl].astype(float).groupby(keys).mean() * 100).add_suffix("_%")
    n = flags["out_any"].groupby(keys).size().rename("n")
    t = pd.concat([n, pct], axis=1)
    return t[t["n"] >= min_n]


def concentration_table(df, flags, key, min_n=50, top=15):
    """¿Los outliers se concentran en pocos juegos / pitchers? (señal de falla de tracking).
    La evidencia más precisa está en las tablas de 'juegos corridos' (por tipo y feature) y 'HorzBreak invertido'."""
    g = flags["out_any"].groupby(df[key]).agg(["sum", "size", "mean"]).rename(columns={"sum": "n_out", "size": "n", "mean": "tasa"})
    g = g[g["n"] >= min_n].sort_values("tasa", ascending=False).head(top)
    base = flags["out_any"].mean()
    g["veces_vs_promedio"] = g["tasa"] / base if base > 0 else np.nan
    return g


# ================================== 8. GRÁFICAS ==================================
def _save(fig, save_dir, name):
    if save_dir:
        import os
        os.makedirs(save_dir, exist_ok=True)
        fig.savefig(os.path.join(save_dir, name), dpi=130, bbox_inches="tight")


def plot_feature_distributions(df, cols, limits=HARD_LIMITS, save_dir=None, max_cols=4):
    cols = [c for c in cols if c in df.columns and df[c].dtype != bool and c not in ("same_hand", "is_lhb", "is_lhp")]
    n = len(cols); nr = int(np.ceil(n / max_cols))
    fig, axes = plt.subplots(nr, max_cols, figsize=(4.3 * max_cols, 3.2 * nr))
    for ax, c in zip(np.ravel(axes), cols):
        x = pd.to_numeric(df[c], errors="coerce").dropna()
        ax.hist(x, bins=80, color="steelblue", alpha=0.85)
        ax.set_yscale("log")
        lo, hi = limits.get(c, (None, None))
        for v in (lo, hi):
            if v is not None and np.isfinite(v):
                ax.axvline(v, color="red", ls="--", lw=1)
        ax.set_title(c, fontsize=10)
    for ax in np.ravel(axes)[n:]:
        ax.axis("off")
    fig.suptitle("Distribución de features (eje Y logarítmico; rojo = límite físico)", y=1.01)
    plt.tight_layout(); _save(fig, save_dir, "01_distribuciones.png"); plt.show()


def plot_movement_outliers(df, flags, pitch_type_col="AutoPitchType", types=None, save_dir=None, sample=40_000):
    types = types or df[pitch_type_col].value_counts().index[:3].tolist()
    fig, axes = plt.subplots(len(types), 2, figsize=(11, 4.2 * len(types)))
    axes = np.atleast_2d(axes)
    for r, pt in enumerate(types):
        m = df[pitch_type_col] == pt
        d = df[m]; f = flags.loc[d.index, "out_any"]
        s = d.sample(min(len(d), sample), random_state=0).index
        for ax, (xc, yc) in zip(axes[r], [("HorzBreak_m", "InducedVertBreak"), ("RelSpeed", "SpinRate")]):
            if xc not in df.columns or yc not in df.columns:
                continue
            ok = ~flags.loc[s, "out_any"]
            ax.scatter(df.loc[s][ok.values][xc], df.loc[s][ok.values][yc], s=3, alpha=0.25, color="gray", label="normal")
            bad = d[f.values]
            ax.scatter(bad[xc], bad[yc], s=14, color="red", alpha=0.8, label=f"marcado ({len(bad)})")
            ax.set_xlabel(xc); ax.set_ylabel(yc); ax.set_title(f"{pt}: {yc} vs {xc}"); ax.legend(markerscale=2, fontsize=8)
    plt.tight_layout(); _save(fig, save_dir, "02_movimiento_y_velocidad.png"); plt.show()


def plot_location_outliers(df, flags, save_dir=None, sample=60_000):
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    s = df.sample(min(len(df), sample), random_state=0).index
    ok = ~flags.loc[s, "out_hard"]
    ax.scatter(df.loc[s][ok.values]["PlateLocSide"], df.loc[s][ok.values]["PlateLocHeight"], s=2, alpha=0.2, color="gray")
    bad = df[flags["out_hard"].values]
    ax.scatter(bad["PlateLocSide"], bad["PlateLocHeight"], s=14, color="red", alpha=0.8, label=f"fuera de límites ({len(bad)})")
    ax.add_patch(plt.Rectangle((-0.83, 1.55), 1.66, 1.95, fill=False, color="black", lw=1.5))
    ax.set_xlabel("PlateLocSide"); ax.set_ylabel("PlateLocHeight"); ax.set_title("Ubicación en el plato y valores imposibles")
    ax.legend(); plt.tight_layout(); _save(fig, save_dir, "03_ubicacion.png"); plt.show()


def plot_flag_rates(df, flags, pitch_type_col="AutoPitchType", altitude_col="altitude_category", save_dir=None):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key in zip(axes, [pitch_type_col, altitude_col, "year"]):
        if key not in df.columns:
            ax.axis("off"); continue
        r = flags[["out_hard", "out_robust", "out_multi", "out_consist", "out_game"]].groupby(df[key]).mean() * 100
        r.plot(kind="bar", ax=ax, width=0.8); ax.set_ylabel("% de pitcheos marcados"); ax.set_title(f"Tasa de outliers por {key}")
        ax.tick_params(axis="x", rotation=45)
    plt.tight_layout(); _save(fig, save_dir, "04_tasas_por_grupo.png"); plt.show()


# ================================== 9. ¿POR QUÉ ESTE STUFF+ ES EXTREMO? ==================================
def explain_extreme_scores(df, score_col, features, pitch_type_col="AutoPitchType", q=0.001, flags=None, top_rows=12):
    """Para el q% más bajo y más alto del score: qué features los explican y cuántos están marcados como outlier."""
    d = df[df[score_col].notna()]
    lo, hi = d[score_col].quantile(q), d[score_col].quantile(1 - q)
    Z = robust_z(d, [c for c in features if c in d.columns and d[c].dtype != bool], pitch_type_col)
    for name, mask in [(f"MÁS BAJOS (≤ {lo:.1f})", d[score_col] <= lo), (f"MÁS ALTOS (≥ {hi:.1f})", d[score_col] >= hi)]:
        print("=" * 70, f"\n{score_col} {name}: {int(mask.sum())} pitcheos\n", "=" * 70, sep="")
        if flags is not None:
            print(f"  % marcados como outlier: {100 * flags.loc[d.index[mask], 'out_any'].mean():.1f}% "
                  f"(base global: {100 * flags['out_any'].mean():.1f}%)")
        print("  Tipos de pitcheo:", d.loc[mask, pitch_type_col].value_counts().head(5).to_dict())
        print("  z robusto promedio por feature (qué los empuja):")
        display(Z[mask].mean().sort_values(key=np.abs, ascending=False).round(2).head(8).to_frame("z_prom"))
        ext = d[mask].assign(_d=(d.loc[mask, score_col] - d[score_col].median()).abs()).sort_values("_d", ascending=False)
        display(ext[[pitch_type_col, score_col] + [c for c in features if c in d.columns][:10]].head(top_rows).round(2))


# ================================== 10. POLÍTICA DE CORRECCIÓN ==================================
def apply_outlier_policy(df, feature_cols, policy="nan_hard", pitch_type_col="AutoPitchType",
                         limits=HARD_LIMITS, winsor_q=(0.001, 0.999)):
    """
    policy:
      "flag"      -> no cambia nada.
      "nan_hard"  -> valores FUERA DE LÍMITE FÍSICO pasan a NaN (error de tracking; XGBoost maneja NaN y el pitcheo
                     se excluye automáticamente si falla una feature núcleo). RECOMENDADA como primer paso.
      "drop_hard" -> elimina las filas con algún valor fuera de límite físico.
      "winsor"    -> primero pone NaN a lo físicamente imposible y luego recorta cada feature a [p0.1, p99.9]
                     dentro de su tipo de pitcheo (conserva el pitcheo).
    No se recomienda borrar outliers solo por ser extremos: los mejores pitcheos son extremos por definición.
    """
    out = df.copy()
    log = {"policy": policy, "filas_antes": len(out)}
    cols = [c for c in feature_cols if c in out.columns and out[c].dtype != bool]
    if policy == "flag":
        return out, log
    if policy in ("nan_hard", "drop_hard"):
        bad_rows = pd.Series(False, index=out.index)
        changed = {}
        for c in cols:
            if c not in limits or c in ("Balls", "Strikes"):
                continue
            lo, hi = limits[c]
            x = pd.to_numeric(out[c], errors="coerce")
            b = (x < lo) | (x > hi)
            changed[c] = int(b.sum())
            bad_rows |= b.fillna(False)
            if policy == "nan_hard":
                out.loc[b.fillna(False), c] = np.nan
        log["valores_corregidos_por_feature"] = {k: v for k, v in changed.items() if v}
        if policy == "drop_hard":
            out = out[~bad_rows].reset_index(drop=True)
    elif policy == "winsor":
        n_clip = {}
        for c in cols:
            x = pd.to_numeric(out[c], errors="coerce")
            if c in limits and c not in ("Balls", "Strikes"):     # 1) errores físicos -> NaN (si no, contaminan el percentil)
                lo_h, hi_h = limits[c]
                x = x.where((x >= lo_h) & (x <= hi_h))
            lo = x.groupby(out[pitch_type_col]).transform(lambda s: s.quantile(winsor_q[0]))
            hi = x.groupby(out[pitch_type_col]).transform(lambda s: s.quantile(winsor_q[1]))
            clipped = x.clip(lo, hi)                              # 2) colas extremas -> percentiles del tipo
            n_clip[c] = int((clipped.ne(pd.to_numeric(out[c], errors="coerce")) & clipped.notna()).sum())
            out[c] = clipped
        log["valores_recortados_por_feature"] = n_clip
    else:
        raise ValueError("policy debe ser flag | nan_hard | drop_hard | winsor")
    log["filas_despues"] = len(out)
    print(log)
    return out, log


def nan_shifted_features(df, flags, game_col="game_anon_id", pitch_type_col="AutoPitchType"):
    """Corrección precisa: pone NaN SOLO a la feature corrida, SOLO en los (juego, tipo de pitcheo) afectados,
    y a HorzBreak en los juegos con signo invertido. El pitcheo se conserva (XGBoost maneja NaN) y se excluye
    del entrenamiento solo si la feature afectada es núcleo (RelSpeed, InducedVertBreak, HorzBreak_m)."""
    out = df.copy(); log = {}
    derived = {"HorzBreak_m": ["hb_diff"], "InducedVertBreak": ["ivb_diff"], "RelSpeed": ["speed_diff"]}
    st = flags.attrs.get("shift_table")
    if st is not None and len(st):
        t = st.reset_index()
        for feat, g in t.groupby("feature"):
            pairs = list(zip(g[game_col], g[pitch_type_col]))
            m = pd.MultiIndex.from_arrays([out[game_col], out[pitch_type_col]]).isin(pairs)
            for c in [feat] + derived.get(feat, []):
                if c in out.columns:
                    out.loc[m, c] = np.nan
            log[f"{feat} (juego x tipo corrido)"] = int(m.sum())
    sg = flags.attrs.get("sign_table")
    if sg is not None and len(sg):
        m = out[game_col].isin(sg.index)
        for c in ["HorzBreak", "HorzBreak_m", "hb_diff"]:
            if c in out.columns:
                out.loc[m, c] = np.nan
        log["HorzBreak (juegos con signo invertido)"] = int(m.sum())
    print("Valores anulados:", log)
    return out, log


def mask_scores_for_flagged(df, flags, score_cols, which="out_core_bad"):
    """No borra filas: solo pone NaN al score de los pitcheos marcados para que no contaminen rankings."""
    out = df.copy()
    for c in score_cols:
        if c in out.columns:
            out.loc[flags[which].reindex(out.index).fillna(False).to_numpy(), c] = np.nan
    return out


# ================================== 11. ORQUESTADOR ==================================
def run_outlier_diagnostics(df, stuff_features, loc_features, pitch_type_col="AutoPitchType",
                            group_col="pitcher_anon_id", game_col="game_anon_id", altitude_col="altitude_category",
                            z_thr=6.0, contamination=0.002, save_dir=None):
    check_basic_integrity(df)

    print("\n" + "=" * 70 + "\n2-5) LÍMITES FÍSICOS, z ROBUSTO, MULTIVARIADO Y CONSISTENCIA\n" + "=" * 70)
    flags, hard_tab, Z = flag_outliers(df, stuff_features, loc_features, pitch_type_col, z_thr, contamination,
                                       game_col, altitude_col, group_col=group_col)
    print("\nFaltantes y fuera de rango por feature:")
    display(hard_tab.round(3))

    tot = len(df)
    print("\nPitcheos marcados:")
    for c in ["out_hard", "out_robust", "out_multi", "out_consist", "out_game", "out_any", "out_core_bad"]:
        print(f"  {c:13s} {int(flags[c].sum()):>9,}  ({100 * flags[c].mean():.3f}%)")
    print("  Con 2+ criterios a la vez:", f"{int((flags['out_n_flags'] >= 2).sum()):,}")

    bg = flags.attrs.get("bad_games")
    if bg is not None and len(bg):
        print(f"\nJuegos con density_index anómalo para su altitud ({len(bg)}):")
        display(bg.round(3).head(15))

    sg = flags.attrs.get("sign_table")
    if sg is not None and len(sg):
        print(f"\nJuegos con HorzBreak INVERTIDO vs los otros juegos de los mismos pitchers ({len(sg)}):")
        display(sg.round(2).head(15))
    print("\nValores pegados a topes / ceros (¿datos recortados o ceros usados como faltante?):")
    display(check_pileups(df).round(3))
    sm = summarize_shift_table(flags)
    if sm is not None:
        print("\nPatrón de los juegos corridos (agrupado por tipo / altitud / feature):")
        display(sm.round(2).head(20))
    st = flags.attrs.get("shift_table")
    if st is not None and len(st):
        print(f"\nJuegos x tipo de pitcheo con promedio corrido vs otros juegos de su altitud ({len(st)}):")
        display(st.round(2).head(15))

    print("\nTasa de marcados por tipo de pitcheo y altitud:")
    display((summarize_flags(df, flags, [pitch_type_col, altitude_col]).round(4)))

    for key in [game_col, group_col]:
        if key in df.columns:
            print(f"\nConcentración por {key} (¿pocos juegos/pitchers explican muchos outliers?):")
            display(concentration_table(df, flags, key).round(4))

    print("\nZ robusto: features con más valores extremos (|z| > %.0f):" % z_thr)
    display((Z.abs() > z_thr).sum().sort_values(ascending=False).to_frame("n_extremos").query("n_extremos > 0"))

    plot_feature_distributions(df, [c for c in dict.fromkeys(list(stuff_features) + list(loc_features))], save_dir=save_dir)
    plot_movement_outliers(df, flags, pitch_type_col, save_dir=save_dir)
    plot_location_outliers(df, flags, save_dir=save_dir)
    plot_flag_rates(df, flags, pitch_type_col, altitude_col, save_dir=save_dir)

    report = {"hard_table": hard_tab, "n_flagged": int(flags["out_any"].sum()), "pct_flagged": float(100 * flags["out_any"].mean())}
    return flags, report

In [ ]:
STUFF_FEATURES = ["RelSpeed", "SpinRate", "SpinAxis_sin", "SpinAxis_cos", "InducedVertBreak", "HorzBreak_m",
                  "RelHeight", "RelSide_m", "Extension", "same_hand"]
LOC_FEATURES = ["PlateLocHeight", "PlateLocSide", "Balls", "Strikes", "is_lhb", "is_lhp"]
flags,report=run_outlier_diagnostics(df,STUFF_FEATURES,LOC_FEATURES)

## 8. Features derivadas

- **Espejo de lateralidad:** todo se lleva a la perspectiva de un pitcher derecho (`HorzBreak`, `RelSide`, `HorzRelAngle` y el eje de giro se voltean en zurdos).
- **`same_hand`:** 1 si bateador y pitcher son de la misma mano.
- **Diferenciales vs fastball principal** (`speed_diff`, `ivb_diff`, `hb_diff`): el fastball principal es el más usado entre Four-Seam/Sinker/TwoSeam/OneSeam; si el pitcher no tiene, se usa su pitcheo más rápido.
- **`density_index`:** densidad relativa del aire por juego, estimada del arrastre medido (1.0 = nivel del mar).

In [ ]:
GRAVITY = 32.174  # ft/s^2
FASTBALL_FAMILY = ["Four-Seam", "Sinker", "TwoSeamFastBall", "OneSeamFastBall"]


def expected_density_ratio(elev_m):
    """Presión relativa vs nivel del mar a igual temperatura (≈ densidad relativa). Referencia física."""
    return (1 - 2.25577e-5 * elev_m) ** 5.25588


def add_mirror_features(df):
    df = df.copy()
    df["is_lhp"] = (df["PitcherThrows"] == "Left").astype(int)
    df["is_lhb"] = (df["BatterSide"] == "Left").astype(int)
    df["same_hand"] = (df["is_lhp"] == df["is_lhb"]).astype(int)
    sign = np.where(df["is_lhp"] == 1, -1.0, 1.0)
    df["HorzBreak_m"] = df["HorzBreak"] * sign
    df["RelSide_m"] = df["RelSide"] * sign
    df["HorzRelAngle_m"] = df["HorzRelAngle"] * sign
    axis_m = np.where(df["is_lhp"] == 1, (360 - df["SpinAxis"]) % 360, df["SpinAxis"])
    df["SpinAxis_sin"] = np.sin(np.deg2rad(axis_m))
    df["SpinAxis_cos"] = np.cos(np.deg2rad(axis_m))
    return df


def add_fastball_diffs(df):
    df = df.copy()
    key = [GROUP_COL, "year"]
    g = (df.groupby(key + [PITCH_TYPE_COL])
           .agg(n=("RelSpeed", "size"), v=("RelSpeed", "mean"),
                ivb=("InducedVertBreak", "mean"), hb=("HorzBreak_m", "mean"))
           .reset_index())
    fam = g[g[PITCH_TYPE_COL].isin(FASTBALL_FAMILY)].sort_values("n", ascending=False).drop_duplicates(key)
    rest = g.merge(fam[key].assign(_has=1), on=key, how="left")
    rest = rest[rest["_has"].isna()].sort_values("v", ascending=False).drop_duplicates(key)
    prim = pd.concat([fam, rest[fam.columns]], ignore_index=True)[key + ["v", "ivb", "hb"]]
    prim = prim.rename(columns={"v": "fb_speed", "ivb": "fb_ivb", "hb": "fb_hb"})
    n0 = len(df)
    df = df.merge(prim, on=key, how="left")
    assert len(df) == n0
    df["speed_diff"] = df["RelSpeed"] - df["fb_speed"]
    df["ivb_diff"] = df["InducedVertBreak"] - df["fb_ivb"]
    df["hb_diff"] = df["HorzBreak_m"] - df["fb_hb"]
    return df.drop(columns=["fb_speed", "fb_ivb", "fb_hb"])


def add_physics_density(df, sea_mean=None):
    """density_index por juego = (arrastre medio del juego) / (arrastre medio a nivel del mar).
    Usa la mediana por juego (más robusta). Devuelve (df, sea_mean) para poder reutilizar la referencia."""
    df = df.copy()
    need = ["vx0", "vy0", "vz0", "ax0", "ay0", "az0"]
    if not all(c in df.columns for c in need):
        print("Aviso: faltan vx0..az0; density_index = 1.0")
        df["density_index"] = 1.0
        return df, sea_mean
    v = np.sqrt(df["vx0"] ** 2 + df["vy0"] ** 2 + df["vz0"] ** 2)
    ux, uy, uz = df["vx0"] / v, df["vy0"] / v, df["vz0"] / v
    azg = df["az0"] + GRAVITY                         # quita gravedad
    drag = -(df["ax0"] * ux + df["ay0"] * uy + azg * uz)   # componente de frenado (positiva)
    df["drag_k"] = drag / (v ** 2)
    df["game_drag_k"] = df.groupby(GAME_COL)["drag_k"].transform("median")
    if sea_mean is None:
        games = df.drop_duplicates(GAME_COL)
        ref = games.loc[games[ALTITUDE_COL] == "No Altitude", "game_drag_k"]
        sea_mean = float(ref.mean() if len(ref) else games["game_drag_k"].mean())
    df["density_index"] = df["game_drag_k"] / sea_mean
    return df, sea_mean


def add_derived_features(df, sea_mean=None):
    df = add_mirror_features(df)
    if USE_FASTBALL_DIFFS:
        df = add_fastball_diffs(df)
    df, sea_mean = add_physics_density(df, sea_mean)
    return df, sea_mean


df, DENSITY_SEA_MEAN = add_derived_features(df)

print("Referencia física (misma temperatura): nivel del mar 1.00 | 540 m %.2f | 1609 m (Denver) %.2f | 2240 m (CDMX) %.2f"
      % (expected_density_ratio(540), expected_density_ratio(1609), expected_density_ratio(2240)))
print("\ndensity_index medido por categoría de altitud:")
display(df.drop_duplicates(GAME_COL).groupby(ALTITUDE_COL)["density_index"].agg(["count", "mean", "std"]).round(3))

## 9. Entrenamiento de Stuff+ por tipo de pitcheo

In [ ]:
XGB_PARAMS = {
    "objective": "multi:softprob", "eta": 0.05, "max_depth": 4, "min_child_weight": 30,
    "subsample": 0.8, "colsample_bytree": 0.8, "lambda": 5.0, "tree_method": "hist", "seed": RANDOM_STATE,
}


def make_group_folds(groups, n_folds, seed=RANDOM_STATE):
    """Asigna cada pitcher a un fold (todos sus pitcheos juntos)."""
    uniq = pd.unique(groups)
    n_folds = max(2, min(n_folds, len(uniq)))
    perm = np.random.default_rng(seed).permutation(len(uniq))
    fold_of = {g: i % n_folds for i, g in enumerate(uniq[perm])}
    return pd.Series(groups).map(fold_of).to_numpy()


def fit_multiclass(X, y, K, rounds=N_ROUNDS, groups=None, es_rounds=20, val_frac=0.1):
    """Entrena XGBoost multiclase. Si se pasan `groups` (pitchers), aparta ~10% de pitchers como
    validación interna y usa parada temprana. Devuelve (modelo, árboles_usados)."""
    params = dict(XGB_PARAMS, num_class=K)
    if groups is None:
        return xgb.train(params, xgb.DMatrix(X, label=y), num_boost_round=rounds), rounds
    uniq = pd.unique(groups)
    rng = np.random.default_rng(RANDOM_STATE)
    val_g = set(rng.choice(uniq, size=max(1, int(len(uniq) * val_frac)), replace=False))
    is_val = pd.Series(groups).isin(val_g).to_numpy()
    dtr = xgb.DMatrix(X[~is_val], label=y[~is_val])
    dva = xgb.DMatrix(X[is_val], label=y[is_val])
    bst = xgb.train(params, dtr, num_boost_round=rounds, evals=[(dva, "val")],
                    early_stopping_rounds=es_rounds, verbose_eval=False)
    return bst, bst.best_iteration + 1


def predict_multiclass(bst, X, K, n_trees=None):
    dm = xgb.DMatrix(X)
    P = bst.predict(dm, iteration_range=(0, n_trees)) if n_trees else bst.predict(dm)
    return P.reshape(len(X), K)


def oof_by_pitcher(sub, features, class_col, classes, train_mask, alt_overrides=None):
    """Predicciones fuera de muestra POR PITCHER.
    - Entrena con las filas train_mask de los pitchers de entrenamiento (con parada temprana).
    - Predice TODAS las filas de los pitchers retenidos (con swing o sin swing).
    - alt_overrides (ej. {"density_index": 1.0}) genera una segunda predicción con esa feature fija.
    - El modelo final (todos los pitchers) usa el promedio de árboles de los folds."""
    K = len(classes)
    cmap = {c: i for i, c in enumerate(classes)}
    y_all = sub[class_col].map(cmap).to_numpy(dtype=float)
    X = sub[features].reset_index(drop=True)
    g_all = sub[GROUP_COL].to_numpy()
    folds = make_group_folds(g_all, N_FOLDS)
    P = np.full((len(sub), K), np.nan)
    P_alt = np.full((len(sub), K), np.nan) if alt_overrides else None
    used = []
    for f in np.unique(folds):
        te = folds == f
        tr = (~te) & train_mask
        bst, n_used = fit_multiclass(X[tr], y_all[tr].astype(int), K, groups=g_all[tr])
        used.append(n_used)
        P[te] = predict_multiclass(bst, X[te], K, n_used)
        if alt_overrides:
            Xa = X[te].copy()
            for c, v in alt_overrides.items():
                Xa[c] = v
            P_alt[te] = predict_multiclass(bst, Xa, K, n_used)
    final, _ = fit_multiclass(X[train_mask], y_all[train_mask].astype(int), K, rounds=max(20, int(np.mean(used))))
    return P, P_alt, final, y_all


def multiclass_report(y, P, K):
    """log-loss del modelo vs log-loss de predecir siempre la frecuencia de cada clase."""
    y = y.astype(int)
    p = np.clip(P[np.arange(len(y)), y], 1e-9, 1)
    prior = np.bincount(y, minlength=K) / len(y)
    base = -np.log(np.clip(prior[y], 1e-9, 1)).mean()
    ll = -np.log(p).mean()
    return ll, base, 100 * (base - ll) / base

In [ ]:
STUFF_FEATURES = ["RelSpeed", "SpinRate", "SpinAxis_sin", "SpinAxis_cos", "InducedVertBreak", "HorzBreak_m",
                  "RelHeight", "RelSide_m", "Extension", "same_hand"]
LOC_FEATURES = ["PlateLocHeight", "PlateLocSide", "Balls", "Strikes", "is_lhb", "is_lhp"]
# df, _ = apply_outlier_policy(df, STUFF_FEATURES + LOC_FEATURES, policy="nan_hard")
# df, _ = nan_shifted_features(df, flags)
# df, _ = apply_outlier_policy(df, STUFF_FEATURES, policy="winsor")   # recorta extremos para que el árbol no extrapole

In [ ]:
STUFF_FEATURES = ["RelSpeed", "SpinRate", "SpinAxis_sin", "SpinAxis_cos", "InducedVertBreak", "HorzBreak_m",
                  "RelHeight", "RelSide_m", "Extension", "same_hand"]
LOC_FEATURES = ["PlateLocHeight", "PlateLocSide", "Balls", "Strikes", "is_lhb", "is_lhp"]

if USE_FASTBALL_DIFFS:
    STUFF_FEATURES += ["speed_diff", "ivb_diff", "hb_diff"]
if USE_DENSITY and "density_index" in df.columns:
    STUFF_FEATURES += ["density_index"]
STUFF_CORE = ["RelSpeed", "InducedVertBreak", "HorzBreak_m"]
print("Features Stuff+:", STUFF_FEATURES)


In [ ]:

print("Features Stuff+:", STUFF_FEATURES)
def candidate_pitch_types(df, min_rows):
    vc = df[PITCH_TYPE_COL].value_counts()
    return [pt for pt, n in vc.items() if pt not in EXCLUDE_PITCH_TYPES and n >= min_rows]


def train_stuff_models(df, features=STUFF_FEATURES, min_swings=MIN_SWINGS_PER_TYPE):
    df = df.copy()
    pcols = [f"P_stuff_{c}" for c in STUFF_CLASSES]
    rcols = [f"Pref_stuff_{c}" for c in STUFF_CLASSES]
    for c in pcols + rcols:
        df[c] = np.nan
    alt = {"density_index": 1.0} if "density_index" in features else None
    base_ok = df["valid_hand"] & df[STUFF_CORE].notna().all(axis=1)
    models, rows = {}, []
    for pt in candidate_pitch_types(df, min_swings):
        t0 = time.time()
        sub = df[(df[PITCH_TYPE_COL] == pt) & base_ok]
        train_mask = (sub["stuff_class"].notna() & sub["has_rv"]).to_numpy()
        if train_mask.sum() < min_swings or sub[GROUP_COL].nunique() < 3:
            print(f"[{pt}] saltado: {int(train_mask.sum())} swings / {sub[GROUP_COL].nunique()} pitchers")
            continue
        P, Pa, final, y_all = oof_by_pitcher(sub, features, "stuff_class", STUFF_CLASSES, train_mask, alt)
        df.loc[sub.index, pcols] = P
        if Pa is not None:
            df.loc[sub.index, rcols] = Pa
        models[pt] = final
        ll, base, imp = multiclass_report(y_all[train_mask], P[train_mask], len(STUFF_CLASSES))
        rows.append({"pitch_type": pt, "swings": int(train_mask.sum()), "pitchers": sub[GROUP_COL].nunique(),
                     "logloss_oof": ll, "logloss_base": base, "mejora_%": imp})
        print(f"[{pt}] listo en {time.time()-t0:.0f}s | mejora de log-loss vs frecuencia base: {imp:.2f}%")
    return df, models, pd.DataFrame(rows)


def finalize_stuff_scores(df, rv_tables, mode=STUFF_RV_MODE):
    df = df.copy()
    pcols = [f"P_stuff_{c}" for c in STUFF_CLASSES]
    rcols = [f"Pref_stuff_{c}" for c in STUFF_CLASSES]
    arr = rv_array(rv_tables["stuff_count"], rv_tables["stuff_neutral"], STUFF_CLASSES)
    neutral = rv_tables["stuff_neutral"].reindex(STUFF_CLASSES).to_numpy(dtype=float)

    def xrv(cols):
        ok = df[cols].notna().all(axis=1)
        out = pd.Series(np.nan, index=df.index)
        if ok.any():
            RV = rv_matrix(df.loc[ok, "Balls"], df.loc[ok, "Strikes"], arr, neutral, mode)
            out[ok] = (df.loc[ok, cols].to_numpy() * RV).sum(axis=1)
        return out

    df["xRV_stuff"] = xrv(pcols)
    ref = xrv(rcols)
    df["xRV_stuff_ref"] = ref.where(ref.notna(), df["xRV_stuff"])
    df["p_whiff_swing"] = df["P_stuff_StrikeSwinging"]

    ok = df["xRV_stuff"].notna()
    norm = {}
    for pt, g in df.loc[ok].groupby(PITCH_TYPE_COL):
        sd = g["xRV_stuff"].std()
        norm[pt] = {"mu": float(g["xRV_stuff"].mean()), "sd": float(sd) if sd and sd > 0 else 1.0}
    mu = df[PITCH_TYPE_COL].map({k: v["mu"] for k, v in norm.items()})
    sd = df[PITCH_TYPE_COL].map({k: v["sd"] for k, v in norm.items()})
    df["Stuff_Plus_PitchType"] = 100 - 10 * (df["xRV_stuff"] - mu) / sd
    df["Stuff_Plus_PitchType_ref"] = 100 - 10 * (df["xRV_stuff_ref"] - mu) / sd
    mu_g, sd_g = df.loc[ok, "xRV_stuff"].mean(), df.loc[ok, "xRV_stuff"].std()
    df["Stuff_Plus"] = 100 - 10 * (df["xRV_stuff"] - mu_g) / sd_g
    return df, norm, {"mu": float(mu_g), "sd": float(sd_g)}


# df, models_stuff, report_stuff = train_stuff_models(df)
# df, norm_stuff, norm_stuff_global = finalize_stuff_scores(df, rv_tables)
# display(report_stuff.round(4))
# df[["xRV_stuff", "Stuff_Plus", "Stuff_Plus_PitchType", "Stuff_Plus_PitchType_ref"]].describe().round(3)

## 10. Entrenamiento de Location+ por tipo de pitcheo

Aquí sí entran **todos** los pitcheos (con y sin swing), porque la ubicación decide bola, strike cantado, swing y contacto.
El valor de cada clase es **por conteo**. Después el xRV se centra por (tipo, conteo) para que sea "ajustado por conteo" (el promedio desde cualquier conteo queda en 0) y se estandariza por tipo.

In [ ]:
LOC_FEATURES = ["PlateLocHeight", "PlateLocSide", "Balls", "Strikes", "is_lhb", "is_lhp"]


def train_location_models(df, features=LOC_FEATURES, min_rows=MIN_PITCHES_LOC):
    df = df.copy()
    pcols = [f"P_loc_{c}" for c in LOC_CLASSES]
    for c in pcols:
        df[c] = np.nan
    base_ok = df["valid_hand"] & df[["PlateLocHeight", "PlateLocSide", "Balls", "Strikes"]].notna().all(axis=1)
    models, rows = {}, []
    for pt in candidate_pitch_types(df, min_rows):
        t0 = time.time()
        sub = df[(df[PITCH_TYPE_COL] == pt) & base_ok]
        train_mask = (sub["loc_class"].notna() & sub["has_rv"]).to_numpy()
        if train_mask.sum() < min_rows or sub[GROUP_COL].nunique() < 3:
            print(f"[{pt}] saltado: {int(train_mask.sum())} pitcheos")
            continue
        P, _, final, y_all = oof_by_pitcher(sub, features, "loc_class", LOC_CLASSES, train_mask)
        df.loc[sub.index, pcols] = P
        models[pt] = final
        ll, base, imp = multiclass_report(y_all[train_mask], P[train_mask], len(LOC_CLASSES))
        rows.append({"pitch_type": pt, "pitcheos": int(train_mask.sum()), "pitchers": sub[GROUP_COL].nunique(),
                     "logloss_oof": ll, "logloss_base": base, "mejora_%": imp})
        print(f"[{pt}] listo en {time.time()-t0:.0f}s | mejora de log-loss vs frecuencia base: {imp:.2f}%")
    return df, models, pd.DataFrame(rows)


def finalize_location_scores(df, rv_tables):
    df = df.copy()
    pcols = [f"P_loc_{c}" for c in LOC_CLASSES]
    arr = rv_array(rv_tables["loc_count"], rv_tables["loc_neutral"], LOC_CLASSES)
    neutral = rv_tables["loc_neutral"].reindex(LOC_CLASSES).to_numpy(dtype=float)
    ok = df[pcols].notna().all(axis=1)
    df["xRV_loc"] = np.nan
    RV = rv_matrix(df.loc[ok, "Balls"], df.loc[ok, "Strikes"], arr, neutral, "by_count")
    df.loc[ok, "xRV_loc"] = (df.loc[ok, pcols].to_numpy() * RV).sum(axis=1)
    # ajuste por conteo y tipo: el promedio desde cualquier conteo queda en 0
    df["xRV_loc_c"] = df["xRV_loc"] - df.groupby([PITCH_TYPE_COL, "Balls", "Strikes"])["xRV_loc"].transform("mean")
    norm = {}
    for pt, g in df.loc[ok].groupby(PITCH_TYPE_COL):
        sd = g["xRV_loc_c"].std()
        norm[pt] = {"mu": float(g["xRV_loc_c"].mean()), "sd": float(sd) if sd and sd > 0 else 1.0}
    mu = df[PITCH_TYPE_COL].map({k: v["mu"] for k, v in norm.items()})
    sd = df[PITCH_TYPE_COL].map({k: v["sd"] for k, v in norm.items()})
    df["Location_Plus"] = 100 - 10 * (df["xRV_loc_c"] - mu) / sd
    return df, norm


df, models_loc, report_loc = train_location_models(df)
df, norm_loc = finalize_location_scores(df, rv_tables)
display(report_loc.round(4))
df[["xRV_loc", "xRV_loc_c", "Location_Plus"]].describe().round(3)

### Guardar modelos y aplicarlos a otro dataset

Guarda los modelos finales (entrenados con todos los pitchers), las tablas de run value y los parámetros de normalización.
`apply_saved_models` sirve para la prueba que planeabas: entrenar con MLB 2024 y aplicar a LMB (o al revés).

In [ ]:
def save_artifacts(models_dir=MODELS_DIR):
    os.makedirs(models_dir, exist_ok=True)
    for pt, m in models_stuff.items():
        m.save_model(os.path.join(models_dir, f"stuff_{pt}.json"))
    for pt, m in models_loc.items():
        m.save_model(os.path.join(models_dir, f"loc_{pt}.json"))
    meta = {
        "stuff_features": STUFF_FEATURES, "loc_features": LOC_FEATURES,
        "stuff_classes": STUFF_CLASSES, "loc_classes": LOC_CLASSES,
        "stuff_types": list(models_stuff), "loc_types": list(models_loc),
        "stuff_rv_mode": STUFF_RV_MODE, "density_sea_mean": DENSITY_SEA_MEAN,
        "norm_stuff": norm_stuff, "norm_stuff_global": norm_stuff_global, "norm_loc": norm_loc,
        "rv_stuff_arr": rv_array(rv_tables["stuff_count"], rv_tables["stuff_neutral"], STUFF_CLASSES).tolist(),
        "rv_stuff_neutral": rv_tables["stuff_neutral"].reindex(STUFF_CLASSES).tolist(),
        "rv_loc_arr": rv_array(rv_tables["loc_count"], rv_tables["loc_neutral"], LOC_CLASSES).tolist(),
        "rv_loc_neutral": rv_tables["loc_neutral"].reindex(LOC_CLASSES).tolist(),
    }
    with open(os.path.join(models_dir, "meta.json"), "w") as f:
        json.dump(meta, f)
    print("Guardado en", models_dir)


def apply_saved_models(df_new, models_dir=MODELS_DIR):
    """Aplica los modelos guardados a un dataset nuevo (sin necesitar resultados ni run values)."""
    meta = json.load(open(os.path.join(models_dir, "meta.json")))
    d = normalize_columns(normalize_dtypes(df_new))
    d, _ = add_derived_features(d, sea_mean=meta["density_sea_mean"])
    base_ok = d["valid_hand"]
    Ks, Kl = len(meta["stuff_classes"]), len(meta["loc_classes"])
    sP = np.full((len(d), Ks), np.nan)
    lP = np.full((len(d), Kl), np.nan)
    for pt in meta["stuff_types"]:
        m = ((d[PITCH_TYPE_COL] == pt) & base_ok & d[STUFF_CORE].notna().all(axis=1)).to_numpy()
        if m.any():
            bst = xgb.Booster(); bst.load_model(os.path.join(models_dir, f"stuff_{pt}.json"))
            sP[m] = predict_multiclass(bst, d.loc[m, meta["stuff_features"]], Ks)
    for pt in meta["loc_types"]:
        m = ((d[PITCH_TYPE_COL] == pt) & base_ok
             & d[["PlateLocHeight", "PlateLocSide", "Balls", "Strikes"]].notna().all(axis=1)).to_numpy()
        if m.any():
            bst = xgb.Booster(); bst.load_model(os.path.join(models_dir, f"loc_{pt}.json"))
            lP[m] = predict_multiclass(bst, d.loc[m, meta["loc_features"]], Kl)

    # Stuff
    ok = ~np.isnan(sP).any(axis=1)
    RV = rv_matrix(d.loc[ok, "Balls"], d.loc[ok, "Strikes"], np.array(meta["rv_stuff_arr"]),
                   np.array(meta["rv_stuff_neutral"]), meta["stuff_rv_mode"])
    d["xRV_stuff"] = np.nan
    d.loc[ok, "xRV_stuff"] = (sP[ok] * RV).sum(axis=1)
    mu = d[PITCH_TYPE_COL].map({k: v["mu"] for k, v in meta["norm_stuff"].items()})
    sd = d[PITCH_TYPE_COL].map({k: v["sd"] for k, v in meta["norm_stuff"].items()})
    d["Stuff_Plus_PitchType"] = 100 - 10 * (d["xRV_stuff"] - mu) / sd
    d["p_whiff_swing"] = sP[:, meta["stuff_classes"].index("StrikeSwinging")]
    # Location
    ok = ~np.isnan(lP).any(axis=1)
    RV = rv_matrix(d.loc[ok, "Balls"], d.loc[ok, "Strikes"], np.array(meta["rv_loc_arr"]),
                   np.array(meta["rv_loc_neutral"]), "by_count")
    d["xRV_loc"] = np.nan
    d.loc[ok, "xRV_loc"] = (lP[ok] * RV).sum(axis=1)
    d["xRV_loc_c"] = d["xRV_loc"] - d.groupby([PITCH_TYPE_COL, "Balls", "Strikes"])["xRV_loc"].transform("mean")
    mu = d[PITCH_TYPE_COL].map({k: v["mu"] for k, v in meta["norm_loc"].items()})
    sd = d[PITCH_TYPE_COL].map({k: v["sd"] for k, v in meta["norm_loc"].items()})
    d["Location_Plus"] = 100 - 10 * (d["xRV_loc_c"] - mu) / sd
    return d


save_artifacts()

## 11. Validación

Con pitchers anonimizados, la validación se hace con **estructura**, no con nombres. Un R² bajo por pitcheo es normal; estas son las pruebas que sí importan:

1. **Calibración por clase:** la probabilidad media predicha debe parecerse a la frecuencia observada (sin `scale_pos_weight` debe cuadrar).
2. **Deciles:** al subir el xRV predicho, el run value real debe subir de forma monótona (pendiente cercana a 1).
3. **Confiabilidad split-half:** el Stuff+ de un pitcher calculado con juegos pares vs impares debe correlacionar. *Ojo:* Stuff+ casi siempre sale muy confiable porque es función de la física del pitcheo; esto **no prueba validez**. Lo que decide si el modelo sirve es la prueba 4 y la mejora de log-loss de las secciones 9 y 10.
4. **Validez predictiva:** el Stuff+ de una mitad debe predecir whiff% y run value *observados* en la otra mitad.
5. **Altitud:** compara Stuff+ a densidad real vs de referencia.

In [ ]:
def calibration_by_class(df, kind="stuff"):
    classes, prefix, class_col = (STUFF_CLASSES, "P_stuff_", "stuff_class") if kind == "stuff" else (LOC_CLASSES, "P_loc_", "loc_class")
    pcols = [prefix + c for c in classes]
    d = df[df[class_col].notna() & df[pcols].notna().all(axis=1)]
    rows = []
    for pt, g in d.groupby(PITCH_TYPE_COL):
        obs = g[class_col].value_counts(normalize=True).reindex(classes).fillna(0).to_numpy()
        pred = g[pcols].mean().to_numpy()
        rows.append(pd.DataFrame({"pitch_type": pt, "clase": classes, "observado": obs, "predicho": pred}))
    out = pd.concat(rows, ignore_index=True)
    out["dif"] = out["predicho"] - out["observado"]
    return out


for kind in ["stuff", "loc"]:
    cal = calibration_by_class(df, kind)
    print(f"\n=== Calibración {kind.upper()}: predicho - observado (puntos porcentuales) ===")
    display((cal.pivot(index="pitch_type", columns="clase", values="dif") * 100).round(2))

In [ ]:
def decile_calibration(df, xrv_col, real_col, mask, title, n_bins=10):
    d = df[mask & df[xrv_col].notna() & df[real_col].notna()].copy()
    d["bin"] = pd.qcut(d[xrv_col], n_bins, labels=False, duplicates="drop")
    t = d.groupby("bin").agg(xrv=(xrv_col, "mean"), real=(real_col, "mean"), n=(xrv_col, "size"))
    slope = np.polyfit(t["xrv"], t["real"], 1)[0] if len(t) > 1 else np.nan
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.plot(t["xrv"], t["real"], "o-")
    lo, hi = min(t["xrv"].min(), t["real"].min()), max(t["xrv"].max(), t["real"].max())
    ax.plot([lo, hi], [lo, hi], "--", color="gray", label="ideal (pendiente 1)")
    ax.set_xlabel("xRV predicho (media del decil)")
    ax.set_ylabel("Run value real (media del decil)")
    ax.set_title(f"{title}\npendiente = {slope:.2f}")
    ax.legend()
    plt.tight_layout(); plt.show()
    return t, slope


# Stuff: solo filas con swing (el modelo es condicional a swing)
mask_sw = df["stuff_class"].notna() & df["has_rv"]
t_stuff, slope_stuff = decile_calibration(df, "xRV_stuff", "delta_run_exp", mask_sw, "Stuff: xRV vs run value real (swings)")

# Location: run value real centrado por (tipo, conteo), igual que el xRV
mask_loc = df["loc_class"].notna() & df["has_rv"]
df["rv_real_c"] = df["delta_run_exp"] - df.groupby([PITCH_TYPE_COL, "Balls", "Strikes"])["delta_run_exp"].transform("mean")
t_loc, slope_loc = decile_calibration(df, "xRV_loc_c", "rv_real_c", mask_loc, "Location: xRV vs run value real (centrado por conteo)")
display(t_stuff.round(4))

In [ ]:
def _add_half(d):
    d = d.copy()
    d["half"] = pd.factorize(d[GAME_COL])[0] % 2     # mitad por paridad de juego
    return d


def split_half_reliability(df, metric_col, min_n=30, by_type=True, mask=None):
    d = df[df[metric_col].notna()] if mask is None else df[mask & df[metric_col].notna()]
    d = _add_half(d)
    keys = [GROUP_COL] + ([PITCH_TYPE_COL] if by_type else [])
    g = d.groupby(keys + ["half"])[metric_col].agg(["mean", "size"]).unstack("half").dropna()
    g = g[(g[("size", 0)] >= min_n) & (g[("size", 1)] >= min_n)]
    rows = []
    groups = g.groupby(level=PITCH_TYPE_COL) if by_type else [("todos", g)]
    for name, gg in groups:
        if len(gg) < 10:
            continue
        r = np.corrcoef(gg[("mean", 0)], gg[("mean", 1)])[0, 1]
        rows.append({"grupo": name, "n_pitchers": len(gg), "r_split_half": r, "r_Spearman_Brown": 2 * r / (1 + r)})
    return pd.DataFrame(rows)


print("Confiabilidad Stuff+ (por pitcher y tipo, min 30 pitcheos por mitad):")
display(split_half_reliability(df, "Stuff_Plus_PitchType").round(3))
print("Confiabilidad Location+ (por pitcher y tipo, min 30 pitcheos por mitad):")
display(split_half_reliability(df, "Location_Plus").round(3))

In [ ]:
def predictive_validity(df, min_pitches=40, min_swings=15):
    """Stuff+ de una mitad de juegos vs resultados observados (en swings) de la otra mitad."""
    d = _add_half(df[df["Stuff_Plus_PitchType"].notna()])
    keys = [GROUP_COL, PITCH_TYPE_COL]
    A = d.groupby(keys + ["half"]).agg(stuff=("Stuff_Plus_PitchType", "mean"), n=("Stuff_Plus_PitchType", "size")).reset_index()
    sw = d[d["stuff_class"].notna() & d["has_rv"]].copy()
    sw["is_ss"] = (sw["stuff_class"] == "StrikeSwinging").astype(float)
    B = sw.groupby(keys + ["half"]).agg(whiff=("is_ss", "mean"), rv=("delta_run_exp", "mean"), ns=("is_ss", "size")).reset_index()
    B["half"] = 1 - B["half"]                    # empareja Stuff de la mitad h con resultados de la otra mitad
    M = A.merge(B, on=keys + ["half"])
    M = M[(M["n"] >= min_pitches) & (M["ns"] >= min_swings)]
    rows = []
    for pt, g in M.groupby(PITCH_TYPE_COL):
        if len(g) >= 20:
            rows.append({"pitch_type": pt, "n_pares": len(g),
                         "corr(Stuff+, whiff/swing)": g["stuff"].corr(g["whiff"]),
                         "corr(Stuff+, RV/swing)": g["stuff"].corr(g["rv"])})
    return pd.DataFrame(rows)


print("Esperado: correlación POSITIVA con whiff/swing y NEGATIVA con run value por swing")
display(predictive_validity(df).round(3))

In [ ]:
# Altitud: Stuff+ con la densidad real de cada juego vs con densidad de referencia (1.0)
alt_tab = (df[df["Stuff_Plus_PitchType"].notna()]
           .groupby(ALTITUDE_COL)
           .agg(pitcheos=("PitchUID", "size"), density_index=("density_index", "mean"),
                stuff_real=("Stuff_Plus_PitchType", "mean"), stuff_ref=("Stuff_Plus_PitchType_ref", "mean"),
                ivb=("InducedVertBreak", "mean"), velo=("RelSpeed", "mean")))
alt_tab["diferencia_ref_menos_real"] = alt_tab["stuff_ref"] - alt_tab["stuff_real"]
display(alt_tab.round(3))

# Movimiento por tipo y altitud (para ver cuánto "pierden" los pitcheos en altura)
mov = (df.groupby([PITCH_TYPE_COL, ALTITUDE_COL])["InducedVertBreak"].mean().unstack(ALTITUDE_COL).round(2))
display(mov)

## 12. Visualizaciones

In [ ]:
top_types = df[PITCH_TYPE_COL].value_counts().index[:2].tolist()

# Distribución de Stuff+ por tipo de pitcheo
fig, ax = plt.subplots(figsize=(10, 5))
d = df[df["Stuff_Plus_PitchType"].notna() & df[PITCH_TYPE_COL].isin(candidate_pitch_types(df, MIN_SWINGS_PER_TYPE))]
sns.boxplot(data=d, x=PITCH_TYPE_COL, y="Stuff_Plus_PitchType", showfliers=False, ax=ax)
ax.axhline(100, color="red", ls="--", alpha=0.6)
ax.set_title("Stuff+ por tipo de pitcheo (cada tipo se compara contra sí mismo)")
plt.tight_layout(); plt.show()

# Location+ en la zona de strike
for pt in top_types:
    plot_heat_map_strike_zone_by_column(df, "Location_Plus", pitch_type=pt)

In [ ]:
# Stuff+ según movimiento
for pt in top_types:
    plot_heat_map_2d(df, x_col="HorzBreak", y_col="InducedVertBreak", target_col="Stuff_Plus_PitchType", pitch_type=pt)

# Cuadrante Stuff+ vs Location+ por pitcher
plot_stuff_vs_location_scatter(df, min_pitches=100)

In [ ]:
# Los mejores pitcheos individuales (Stuff+ > 130) en la zona
mejores = df[df["Stuff_Plus_PitchType"] > 130]
print("Pitcheos con Stuff+ > 130:", len(mejores))
if len(mejores):
    plot_pitches(mejores.head(5000))

## 13. Rankings de pitchers

Las funciones usan por defecto `Stuff_Plus_PitchType` (cada pitcheo calificado contra su propio tipo), porque el Stuff+ global mezcla tipos de pitcheo y favorece a quien lanza más rectas.
Para evaluar a un pitcher en CDMX sin que la altura lo penalice, usa `metric_col="Stuff_Plus_PitchType_ref"`.

In [ ]:
def get_pitcher_arsenal(df_data, pitcher_id, min_pitches=10, metric_col="Stuff_Plus_PitchType"):
    p_df = df_data[df_data[GROUP_COL] == pitcher_id]
    if len(p_df) == 0:
        print(f"Pitcher {pitcher_id} no encontrado.")
        return None
    total_p = len(p_df)
    arsenal = (p_df.groupby(PITCH_TYPE_COL)
               .agg(n_pitches=("PitchUID", "count"), velo=("RelSpeed", "mean"),
                    ivb=("InducedVertBreak", "mean"), hb=("HorzBreak", "mean"),
                    spin=("SpinRate", "mean"), xwhiff_swing=("p_whiff_swing", "mean"),
                    stuff_plus=(metric_col, "mean"), location_plus=("Location_Plus", "mean"))
               .reset_index())
    arsenal = arsenal[arsenal["n_pitches"] >= min_pitches].copy()
    arsenal["usage_pct"] = arsenal["n_pitches"] / total_p * 100
    arsenal = arsenal.sort_values("usage_pct", ascending=False)
    w = arsenal["n_pitches"] / arsenal["n_pitches"].sum()
    print(f"\n{'='*70}\n ARSENAL {pitcher_id} | pitcheos: {total_p} | "
          f"Stuff+ overall: {(arsenal['stuff_plus']*w).sum():.1f} | Location+ overall: {(arsenal['location_plus']*w).sum():.1f}\n{'='*70}")
    show = arsenal.copy()
    show["usage_pct"] = show["usage_pct"].map("{:.1f}%".format)
    show["velo"] = show["velo"].map("{:.1f}".format)
    show["ivb"] = show["ivb"].map("{:+.1f}".format)
    show["hb"] = show["hb"].map("{:+.1f}".format)
    show["xwhiff_swing"] = (show["xwhiff_swing"] * 100).map("{:.1f}%".format)
    show["stuff_plus"] = show["stuff_plus"].round(1)
    show["location_plus"] = show["location_plus"].round(1)
    display(show[[PITCH_TYPE_COL, "usage_pct", "n_pitches", "velo", "ivb", "hb", "xwhiff_swing", "stuff_plus", "location_plus"]])
    return arsenal


def get_top_pitchers_by_environment(df_data, n=10, min_pitches=100, names_map=None, metric_col="Stuff_Plus_PitchType"):
    """Top N por Stuff+ a nivel del mar (No Altitude) y en CDMX (Extreme Altitude)."""
    envs = {"Nivel del mar": "No Altitude", "Harp Helú / CDMX (altitud extrema)": "Extreme Altitude"}
    results = {}
    for env_name, key in envs.items():
        e = df_data[df_data[ALTITUDE_COL] == key]
        if len(e) == 0:
            print(f"Sin datos para {key}")
            continue
        agg = (e.groupby(GROUP_COL)
               .agg(pitcheos=("PitchUID", "count"), stuff_plus=(metric_col, "mean"),
                    location_plus=("Location_Plus", "mean"), velo=("RelSpeed", "mean"),
                    ivb=("InducedVertBreak", "mean"), hb=("HorzBreak", "mean"),
                    xwhiff_swing=("p_whiff_swing", "mean"))
               .query("pitcheos >= @min_pitches").sort_values("stuff_plus", ascending=False).head(n).reset_index())
        agg["nombre"] = agg[GROUP_COL].map(names_map).fillna("Desconocido")
        agg["xwhiff_swing"] = (agg["xwhiff_swing"] * 100).round(1).astype(str) + "%"
        print(f"\n{'='*70}\n TOP {n}: {env_name} (mín. {min_pitches} pitcheos) | métrica: {metric_col}\n{'='*70}")
        display(agg.round(1))
        results[key] = agg
    return results


def get_best_pitchers_by_pitch_type(df_data, top_per_type=3, min_pitches=50, environment=None,
                                    names_map=None, metric_col="Stuff_Plus_PitchType"):
    data = df_data if environment is None else df_data[df_data[ALTITUDE_COL] == environment]
    agg = (data.groupby([PITCH_TYPE_COL, GROUP_COL])
           .agg(pitcheos=("PitchUID", "count"), stuff_plus=(metric_col, "mean"),
                location_plus=("Location_Plus", "mean"), velo=("RelSpeed", "mean"),
                ivb=("InducedVertBreak", "mean"), hb=("HorzBreak", "mean"),
                xwhiff_swing=("p_whiff_swing", "mean"))
           .reset_index())
    agg = agg[(agg["pitcheos"] >= min_pitches) & agg["stuff_plus"].notna()].copy()
    agg["nombre"] = agg[GROUP_COL].map(names_map).fillna("Desconocido")
    agg = agg.sort_values([PITCH_TYPE_COL, "stuff_plus"], ascending=[True, False])
    best = agg.groupby(PITCH_TYPE_COL).head(top_per_type).reset_index(drop=True)
    best["xwhiff_swing"] = (best["xwhiff_swing"] * 100).round(1).astype(str) + "%"
    print(f"\nMEJORES POR TIPO ({'liga completa' if environment is None else environment} | top {top_per_type} | mín. {min_pitches})")
    display(best.round(1))
    return best

In [ ]:
top_general = get_top_pitchers_by_environment(df, n=10, min_pitches=100)
mejores_por_tipo = get_best_pitchers_by_pitch_type(df, top_per_type=3, min_pitches=50)
mejores_cdmx = get_best_pitchers_by_pitch_type(df, top_per_type=3, min_pitches=50, environment="Extreme Altitude",
                                               metric_col="Stuff_Plus_PitchType_ref")

In [ ]:
# Arsenal de un pitcher en particular
ejemplo = df[GROUP_COL].value_counts().index[0]
_ = get_pitcher_arsenal(df, ejemplo)

## Guardar resultados

In [ ]:
"""
DIAGNÓSTICO DE STUFF+: ¿está "viendo" la ubicación? ¿qué hay en las colas? ¿ranking de pocos pitcheos?
=========================================================================================================
Uso (después de entrenar Stuff+ en el notebook):

    exec(open("diagnostico_stuff.py", encoding="utf-8").read())

    # 1) ¿Las features de Stuff+ permiten reconstruir la ubicación? (fuga)
    leak = location_leakage_check(df, STUFF_FEATURES)
    # 2) ¿Stuff+ correlaciona con la altura/lado donde cae el pitcheo?
    stuff_location_corr(df)
    # 3) ¿Cómo son los pitcheos con Stuff+ > 130?
    top_tail_profile(df, "Stuff_Plus_PitchType", thr=130, flags=flags)
    # 4) Ranking de pitchers con encogimiento hacia el promedio (para que pocos pitcheos no dominen)
    rank = shrink_pitcher_means(df, "Stuff_Plus_PitchType", keys=("pitcher_anon_id", "AutoPitchType"))
"""
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display


def _group_folds(groups, n_folds=2, seed=42):
    uniq = pd.unique(groups)
    perm = np.random.default_rng(seed).permutation(len(uniq))
    fold_of = {g: i % n_folds for i, g in enumerate(uniq[perm])}
    return pd.Series(groups).map(fold_of).to_numpy()


# ============================ 1. FUGA DE UBICACIÓN ============================
def location_leakage_check(df, features, target="PlateLocHeight", pitch_type_col="AutoPitchType",
                           group_col="pitcher_anon_id", types=None, sample=60_000, seed=42):
    """Predice la ubicación (altura o lado) SOLO con las features de Stuff+ (validación por pitcher).
    R² alto = las features contienen información de ubicación (fuga). Compara varios subconjuntos para ver
    qué features la causan. El 'piso' (velocidad+movimiento+spin) es la relación física normal movimiento-altura."""
    release_angles = [c for c in ["VertRelAngle", "HorzRelAngle", "HorzRelAngle_m"] if c in features]
    release_pos = [c for c in ["RelHeight", "RelSide_m", "RelSide", "Extension"] if c in features]
    floor = [c for c in ["RelSpeed", "SpinRate", "SpinAxis_sin", "SpinAxis_cos", "InducedVertBreak", "HorzBreak_m"] if c in features]
    sets = {"todas las features": features,
            "sin ángulos de salida": [f for f in features if f not in release_angles],
            "sin ángulos ni posición de salida": [f for f in features if f not in release_angles + release_pos],
            "solo velocidad+movimiento+spin (piso)": floor}
    types = types or df[pitch_type_col].value_counts().index[:3].tolist()
    params = {"objective": "reg:squarederror", "eta": 0.1, "max_depth": 5, "min_child_weight": 20,
              "subsample": 0.8, "tree_method": "hist", "seed": seed}
    rows = []
    for pt in types:
        d = df[df[pitch_type_col] == pt].dropna(subset=[target])
        if len(d) > sample:
            d = d.sample(sample, random_state=seed)
        folds = _group_folds(d[group_col].to_numpy(), 2, seed)
        y = d[target].to_numpy(dtype=float)
        for name, feats in sets.items():
            if not feats:
                continue
            pred = np.full(len(d), np.nan)
            for f in (0, 1):
                te, tr = folds == f, folds != f
                bst = xgb.train(params, xgb.DMatrix(d.loc[tr, feats], label=y[tr]), 150)
                pred[te] = bst.predict(xgb.DMatrix(d.loc[te, feats]))
            r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
            rows.append({"tipo": pt, "target": target, "conjunto": name, "n_features": len(feats), "R2_ubicacion": r2})
    out = pd.DataFrame(rows).pivot_table(index=["tipo", "target"], columns="conjunto", values="R2_ubicacion")
    out = out[[c for c in sets if c in out.columns]]
    print(f"R² para predecir {target} con las features de Stuff+ (validación por pitcher):")
    display(out.round(3))
    print("Lectura: si 'todas' >> 'sin ángulos de salida', los ángulos de salida están filtrando la ubicación al Stuff+.")
    return out


def stuff_location_corr(df, score_col="Stuff_Plus_PitchType", pitch_type_col="AutoPitchType", q=0.99):
    """Correlación de Stuff+ con la altura/lado real del pitcheo, por tipo."""
    rows = []
    for pt, g in df.groupby(pitch_type_col):
        g = g.dropna(subset=[score_col, "PlateLocHeight", "PlateLocSide"])
        if len(g) < 1000:
            continue
        top = g[score_col] >= g[score_col].quantile(q)
        rows.append({"tipo": pt, "n": len(g),
                     "corr(Stuff+, altura)": g[score_col].corr(g["PlateLocHeight"]),
                     "corr(Stuff+, |lado|)": g[score_col].corr(g["PlateLocSide"].abs()),
                     "altura_media_top1%": g.loc[top, "PlateLocHeight"].mean(),
                     "altura_media_resto": g.loc[~top, "PlateLocHeight"].mean(),
                     "%_top1%_bajo_1.5ft": 100 * (g.loc[top, "PlateLocHeight"] < 1.5).mean(),
                     "%_resto_bajo_1.5ft": 100 * (g.loc[~top, "PlateLocHeight"] < 1.5).mean()})
    out = pd.DataFrame(rows)
    display(out.round(3))
    print("Lectura: correlación negativa con altura = Stuff+ premia pitcheos que caen bajos (ubicación disfrazada de stuff).")
    return out


# ============================ 2. COLAS ============================
def top_tail_profile(df, score_col="Stuff_Plus_PitchType", thr=130, pitch_type_col="AutoPitchType",
                     group_col="pitcher_anon_id", flags=None):
    tail = df[df[score_col] >= thr]
    print(f"Pitcheos con {score_col} >= {thr}: {len(tail):,} de {df[score_col].notna().sum():,} "
          f"({100 * len(tail) / max(df[score_col].notna().sum(), 1):.3f}%)")
    comp = pd.DataFrame({"% en la cola": tail[pitch_type_col].value_counts(normalize=True) * 100,
                         "% del total": df.loc[df[score_col].notna(), pitch_type_col].value_counts(normalize=True) * 100}).fillna(0)
    comp["veces"] = comp["% en la cola"] / comp["% del total"].replace(0, np.nan)
    print("\nComposición por tipo (veces > 1 = sobrerrepresentado en la cola):")
    display(comp.sort_values("% en la cola", ascending=False).round(2))
    if flags is not None:
        print(f"\n% de la cola marcado como outlier: {100 * flags.loc[tail.index, 'out_any'].mean():.1f}% "
              f"(global: {100 * flags['out_any'].mean():.1f}%)")
    cols = [c for c in ["RelSpeed", "InducedVertBreak", "HorzBreak_m", "SpinRate", "RelHeight", "VertRelAngle",
                        "PlateLocHeight", "PlateLocSide"] if c in df.columns]
    prof = pd.DataFrame({"cola": tail[cols].median(), "mismo tipo (mediana)": df[df[pitch_type_col].isin(tail[pitch_type_col].unique())][cols].median()})
    print("\nPerfil de la cola vs mediana de los mismos tipos:")
    display(prof.round(2))
    print("\nPitchers que más aparecen en la cola:")
    display(tail[group_col].value_counts().head(8).to_frame("pitcheos_en_cola"))


# ============================ 3. RANKING CON ENCOGIMIENTO ============================
def shrink_pitcher_means(df, metric="Stuff_Plus_PitchType", keys=("pitcher_anon_id",), min_n=20):
    """Encogimiento empírico (Bayes empírico) de la media por pitcher hacia la media global.
    shrink = tau² / (tau² + sigma²/n), con tau² = varianza real entre pitchers y sigma² = varianza dentro."""
    keys = list(keys)
    g = df.dropna(subset=[metric]).groupby(keys)[metric].agg(media="mean", var_dentro="var", n="size").reset_index()
    g = g[g["n"] >= min_n].copy()
    sigma2 = np.average(g["var_dentro"].fillna(0), weights=g["n"] - 1)
    grand = np.average(g["media"], weights=g["n"])
    var_means = np.average((g["media"] - grand) ** 2, weights=g["n"])
    tau2 = max(var_means - np.average(sigma2 / g["n"], weights=g["n"]), 1e-9)
    g["factor"] = tau2 / (tau2 + sigma2 / g["n"])
    g["media_encogida"] = grand + g["factor"] * (g["media"] - grand)
    print(f"Varianza dentro del pitcher (sigma²): {sigma2:.2f} | varianza real entre pitchers (tau²): {tau2:.2f}")
    print(f"Factor de encogimiento: mediana {g['factor'].median():.3f}, mínimo {g['factor'].min():.3f} "
          f"(1.0 = sin encogimiento; los pitcheos de un mismo pitcher no son independientes, así que el efecto suele ser pequeño)")
    return g.sort_values("media_encogida", ascending=False)

In [ ]:
location_leakage_check(df,STUFF_FEATURES)
top_tail_profile(df)
shrink_pitcher_means(df)

## Visualizacion relacion stuff plus era


In [ ]:
eraInfo = pd.read_csv("pitcher_era_por_anio2.csv")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

eraInfo = pd.read_csv('pitcher_era_por_anio2.csv') 
min_ip = 50
eraInfo = eraInfo[eraInfo['IP']>min_ip]
x = df.groupby(GROUP_COL)['Stuff_Plus'].mean()
y = eraInfo.groupby(GROUP_COL)['ERA'].mean()



datos_combinados = pd.concat([x, y], axis=1).dropna()
plt.scatter(datos_combinados['Stuff_Plus'], datos_combinados['ERA'], alpha=0.7,)

coeficientes = np.polyfit(datos_combinados['Stuff_Plus'], datos_combinados['ERA'], 1)
polinomio = np.poly1d(coeficientes)

# Generamos puntos en el eje X (desde el mínimo Stuff+ hasta el máximo) para trazar la línea
x_linea = np.linspace(datos_combinados['Stuff_Plus'].min(), datos_combinados['Stuff_Plus'].max(), 100)
y_linea = polinomio(x_linea)

# Dibujamos la recta punteada (ls='--') de color rojo (color='red')
plt.plot(x_linea, y_linea, color='red', linestyle='--', linewidth=2, label='Tendencia (Inversa)')
for nombre_pitcher, fila in datos_combinados.iterrows():
    plt.annotate(
        nombre_pitcher,                              # El ID del pitcher ahora es el índice (la clave de la fila)
        (fila['Stuff_Plus'], fila['ERA']),           # Coordenadas X, Y
        textcoords="offset points", 
        xytext=(5, 5), 
        ha='left', 
        fontsize=8,
        alpha=0.8
    )
plt.title('Relación entre Stuff+ y ERA')
plt.xlabel('Stuff+ (Promedio)')
plt.ylabel('ERA (Promedio)')
plt.grid(True)

# 6. Mostrar la gráfica
plt.show()


## 14. Experimentos: targets, valores de carrera y features (MLflow)

Requiere `experimentos_stuff.py` en la misma carpeta y `pip install mlflow`. **No reentrena ni modifica** lo de arriba: toma `df` ya limpio (secciones 1-8 y política de outliers) y corre una rejilla de versiones.

| Eje | Qué cambia | Dónde está |
|---|---|---|
| `target` | clases y filas de entrenamiento (solo swings, o todos los pitcheos) | `SCHEMES` |
| `rv` / `rv_mult` | cuánto vale cada clase (neutral, por conteo, por entorno, solo whiff, multiplicadores para whiff/contacto débil/chase) | `RV_KINDS` |
| `features` | C1 densidad, C2 arsenal por entorno, spin, VAA, movimiento por rpm | `FEATURE_SETS` |
| `params` | profundidad, regularización, tasa de aprendizaje | `PARAM_SETS` |

**Comparación:** el log-loss solo compara corridas con el mismo `target`. Entre targets distintos usar `auc_whiff`, `calib_slope`, `next_whiff_r_stuff`, `s6_r_dstuff_dwhiff` y `xrv_vs_observed_r`.


In [ ]:
#%pip install mlflow
import importlib, experimentos_stuff as ex
importlib.reload(ex)


In [ ]:
import pandas as pd
df = pd.read_parquet("stuff_location_scoredggg.parquet")

In [ ]:

df = ex.prepare_extra_features(df)      # alt_ord, total_break, move_per_rpm, vaa_est, diffs de arsenal por entorno
print([c for c in ["alt_ord","total_break","move_per_rpm","vaa_est","speed_diff_env","ivb_diff_env","hb_diff_env"] if c in df.columns])


In [ ]:
print(df[
    ["total_break", "move_per_rpm", "vaa_est",
     "speed_diff_env", "ivb_diff_env", "hb_diff_env"]
].dtypes)

In [ ]:
# Corrida única (reproduce el Stuff+ actual) para ver todo lo que se guarda en runs/<nombre>/
m, out = ex.run_experiment(df, ex.E(name="T0_swing6_neutral_full", target="swing6", rv="neutral",
                                    save_oof=True, shap_sample=3000, fit_final=True),
                           dataset_path=DATA_PATH)
ex.plot_learning_curves(out["curves"], "T0", show=True)
ex.plot_importance(out["importance"], title="T0", show=True)
out["folds"]       # árboles usados por fold, train vs val, ¿paró temprano?


In [ ]:
# Cribado rápido: 3 tipos (Four-Seam, Slider, Changeup) y 50% de lanzadores. Bloque 1: target y valor de carrera.
summary_t = ex.run_grid(df, ex.GRID_TARGETS, screen=True, dataset_path=DATA_PATH)
ex.compare_runs()


In [ ]:
# Bloque 2: features (C1/C2/H4) y bloque 3: hiperparámetros, con el target actual
summary_f = ex.run_grid(df, ex.GRID_FEATURES + ex.GRID_PARAMS, screen=True, dataset_path=DATA_PATH)
ex.compare_runs()


In [ ]:
# Rejilla propia: producto cartesiano
mi_grid = ex.make_grid(targets=["swing6", "swing_cq"], rvs=["neutral", "by_env"], features=["base", "base_arsenal_env"])
# summary = ex.run_grid(df, mi_grid, screen=True, dataset_path=DATA_PATH)

# Run value propio: multiplicadores por clase (ej. premiar whiff y contacto débil)
ex.run_experiment(df, ex.E(name="mi_rv", target="swing_cq", rv="neutral", rv_mult={"Whiff": 2.0, "Weak": 1.5}))

# Finalistas con datos completos (sin screen): se repiten los 2-3 mejores
ex.run_grid(df, [ex.E(name="FINAL_A", target="swing_cq", rv="by_env", features="base_arsenal_env", fit_final=True, save_oof=True)],
            screen=False, dataset_path=DATA_PATH)

# Interfaz: en una terminal, desde esta carpeta ->  mlflow ui --backend-store-uri sqlite:///mlflow.db


In [ ]:
import importlib, experimentos_stuff as ex
importlib.reload(ex)



In [ ]:
grid = [{**g, "save_oof": True} for g in ex.GRID_NEXT]
DATA_PATH = "stuff_location_scoredggg.parquet"
summary = ex.run_grid(df, grid, screen=False, dataset_path=DATA_PATH)

a = ex.load_oof("N1_swingcq_arsenalenv")
b = ex.load_oof("T0_swing6_neutral_full")
ex.bootstrap_metrics(a, b, n_boot=300)   # columna 'sig' = True si la diferencia es real

In [ ]:
import importlib, experimentos_stuff as ex
importlib.reload(ex)

n1 = ex.load_oof("N1_swingcq_arsenalenv")
n2 = ex.load_oof("N2_swingcq_arsenalenv_handneutral")
n3 = ex.load_oof("N3_swingcq_arsenalenv_nodensity")
t0 = ex.load_oof("T0_swing6_neutral_full")

print("N1 vs T0 (¿swing_cq + arsenal_env mejora el baseline?)")
print(ex.bootstrap_metrics(n1, t0, n_boot=300).to_string())
print("N3 vs N1 (C1: ¿quitar la densidad?)")
print(ex.bootstrap_metrics(n3, n1, n_boot=300).to_string())
print("N2 vs N1 (¿mano neutral cambia el ranking?)")
print(ex.bootstrap_metrics(n2, n1, n_boot=300).to_string())

In [ ]:
final = ex.E(name="FINAL_v1", target="swing_cq", rv="neutral",
             features="arsenal_env_nodensity", hand_neutral=True, params="base",
             fit_final=True, save_oof=True, shap_sample=3000,
             question="C1", hypothesis="propia", status="candidate",
             report_section="4-modelado", product_claim="stuff-plus-significa-algo")
m, out = ex.run_experiment(df, final, dataset_path=DATA_PATH)

# comparación pareada contra el baseline y contra N1
f = ex.load_oof("FINAL_v1")
print(ex.bootstrap_metrics(f, ex.load_oof("T0_swing6_neutral_full"), n_boot=300))
print(ex.bootstrap_metrics(f, ex.load_oof("N1_swingcq_arsenalenv"), n_boot=300))

In [ ]:
"""
FINALISTAS — entrena y compara los candidatos con datos completos
=================================================================
Uso en el notebook (df ya limpio y con ex.prepare_extra_features aplicado):

    import importlib, experimentos_stuff as ex, finalistas as fin
    importlib.reload(ex); importlib.reload(fin)
    resumen, comparaciones = fin.main(df, dataset_path=DATA_PATH)

Candidatos (todos: valor de carrera neutral, features arsenal_env sin densidad, hand_neutral, params base)
  FINAL_v1        swing_cq : P(resultado | swing) con clase "contacto débil"   (el que ya corriste; se omite si existe en MLflow)
  FIN_B_all8      all8     : todos los pitcheos (Ball/StrikeCalled incluidos) sin ubicación
  FIN_C_allcq     all_cq   : como all8 + clase "contacto débil"

OJO al leer las métricas:
  * En all8/all_cq el Stuff+ es valor POR PITCHEO (incluye si el bateador se la deja pasar o le hace swing);
    en swing_cq es valor POR SWING. Se comparan con las mismas métricas de lanzador, pero no son la misma cantidad.
  * El log-loss no es comparable entre targets distintos: usar auc_whiff y las métricas de lanzador.
"""
import pandas as pd
import experimentos_stuff as ex

COMMON = dict(rv="neutral", features="arsenal_env_nodensity", hand_neutral=True, params="base",
              fit_final=True, save_oof=True, shap_sample=3000, status="candidate",
              report_section="4-modelado", product_claim="stuff-plus-significa-algo", question="A1")

FINALISTAS = [
    ex.E(name="FINAL_v1",    target="swing_cq", **{**COMMON, "question": "C1"}),
    ex.E(name="FIN_B_all8",  target="all8",     **COMMON),
    ex.E(name="FIN_C_allcq", target="all_cq",   **COMMON),
]
BASELINES = ["T0_swing6_neutral_full"]            # corrida de referencia (la que ya tienes en runs/)


def mainFin(df, dataset_path=None, n_boot=300, overrides=None, out_dir="runs", use_mlflow=True,
         tracking_uri="sqlite:///mlflow.db", experiment_name="superestela/A-target/stuff-grid"):
    grid = [{**g, **(overrides or {})} for g in FINALISTAS]
    resumen = ex.run_grid(df, grid, screen=False, dataset_path=dataset_path, out_dir=out_dir,
                          use_mlflow=use_mlflow, tracking_uri=tracking_uri, experiment_name=experiment_name)
    cols = ["run", "auc_whiff", "calib_slope", "calib_slope_extreme", "calib_slope_sea", "splithalf_r_mean",
            "next_whiff_r_stuff", "next_rv_r_stuff", "xrv_vs_observed_r", "xrv_next_season_rv_r",
            "s6_r_dstuff_dwhiff", "n_trees_mean", "val_minus_train_logloss", "runtime_s"]
    if len(resumen):
        print(resumen[[c for c in cols if c in resumen.columns]].round(3).to_string(index=False))

    oof = {}
    for nombre in [g["name"] for g in grid] + BASELINES:
        try:
            oof[nombre] = ex.load_oof(nombre, out_dir)
        except Exception as e:
            print(f"(sin oof de {nombre}: {e})")
    comparaciones = {}
    ref = oof.get("FINAL_v1")
    for nombre, res in oof.items():
        if nombre == "FINAL_v1" or ref is None:
            continue
        print(f"\n{nombre}  vs  FINAL_v1   (diff > 0 = {nombre} mayor; sig=True: el IC95% no cruza 0)")
        comparaciones[nombre] = ex.bootstrap_metrics(res, ref, n_boot=n_boot)
        print(comparaciones[nombre][["metric", "est", "est_other", "diff", "diff_lo", "diff_hi", "sig"]].to_string(index=False))
    return resumen, comparaciones


def stuff_a_df(df, nombre, columna=None, out_dir="runs"):
    """Agrega al df el Stuff+ fuera de muestra de una corrida (por PitchUID)."""
    o = ex.load_oof(nombre, out_dir)[["PitchUID", "stuff"]].rename(columns={"stuff": columna or f"Stuff_{nombre}"})
    return df.merge(o, on="PitchUID", how="left")


In [ ]:
import importlib, experimentos_stuff as ex
importlib.reload(ex)

resumen, comparaciones = mainFin(df, dataset_path=DATA_PATH)

In [ ]:
#df = stuff_a_df(df, "FIN_B_all8", columna="Stuff_all8")

## 15. Scoring y rankings desde el run `FINAL_v1`

Esta celda carga los artefactos ya registrados en MLflow; no reentrena ni ejecuta las secciones anteriores.


In [ ]:
# Solo inferencia: carga FINAL_v1, calcula las features disponibles y crea rankings.
from pathlib import Path
import json
import tempfile
import numpy as np
import pandas as pd
import xgboost as xgb
import experimentos_stuff as ex

try:
    display
except NameError:
    from IPython.display import display

RUN_NAME = "FIN_B_all8"
TRACKING_URI = "sqlite:///mlflow.db"
EXPERIMENT_NAME = "superestela/A-target/stuff-grid"
RUNS_DIR = Path("runs")
INPUT_DATA_PATH = "stuff_model_df.parquet"
SCORE_COLUMN = "StuffALL"


def _artifact_paths(run_name: str) -> dict[str, Path]:
    """Resolve a local run or download the required artifacts from MLflow."""
    candidates = [
        RUNS_DIR / run_name,
        Path("mlFlow") / "runs" / run_name,
        Path("..") / "mlFlow" / "runs" / run_name,
    ]
    for local in candidates:
        required = [local / "config.json", local / "rv_values.csv", local / "oof.parquet"]
        if all(path.exists() for path in required):
            return {"root": local, "config": required[0], "rv": required[1], "oof": required[2]}

    try:
        import mlflow
        from mlflow.tracking import MlflowClient
    except ImportError as error:
        raise RuntimeError("MLflow no est? instalado y no existe el run local requerido.") from error

    mlflow.set_tracking_uri(TRACKING_URI)
    client = MlflowClient()
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment is None:
        raise FileNotFoundError(f"No existe el experimento MLflow: {EXPERIMENT_NAME}")
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
        order_by=["attributes.start_time DESC"],
        max_results=1,
    )
    if not runs:
        raise FileNotFoundError(f"No se encontr? el run MLflow: {run_name}")

    root = Path(tempfile.mkdtemp(prefix=f"{run_name}_"))
    paths = {"root": root}
    for artifact in ("config.json", "rv_values.csv", "oof.parquet"):
        paths[artifact.removesuffix(".csv").removesuffix(".json").removesuffix(".parquet")] = Path(
            client.download_artifacts(runs[0].info.run_id, artifact, str(root))
        )
    paths["run_id"] = Path(runs[0].info.run_id)
    return paths


def _prepare_run_features(frame: pd.DataFrame, required: list[str]) -> pd.DataFrame:
    """Build derived features when their inputs exist; XGBoost handles remaining NaNs."""
    out = frame.copy()
    for column in ["SpinAxis", "HorzBreak", "RelSide", "RelSpeed", "SpinRate",
                   "InducedVertBreak", "RelHeight", "Extension"]:
        if column in out.columns:
            out[column] = pd.to_numeric(out[column], errors="coerce")
    if "PitcherThrows" in out.columns and "BatterSide" in out.columns:
        out["is_lhp"] = out["PitcherThrows"].eq("Left").astype("int8")
        out["is_lhb"] = out["BatterSide"].eq("Left").astype("int8")
        out["same_hand"] = (out["is_lhp"] == out["is_lhb"]).astype("int8")
        sign = np.where(out["is_lhp"].eq(1), -1.0, 1.0)
        if "HorzBreak" in out.columns:
            out["HorzBreak_m"] = out["HorzBreak"] * sign
        if "RelSide" in out.columns:
            out["RelSide_m"] = out["RelSide"] * sign
        if "SpinAxis" in out.columns:
            axis_m = np.where(out["is_lhp"].eq(1), (360.0 - out["SpinAxis"]) % 360.0, out["SpinAxis"])
            out["SpinAxis_sin"] = np.sin(np.deg2rad(axis_m))
            out["SpinAxis_cos"] = np.cos(np.deg2rad(axis_m))
    elif "SpinAxis" in out.columns:
        out["SpinAxis_sin"] = np.sin(np.deg2rad(out["SpinAxis"]))
        out["SpinAxis_cos"] = np.cos(np.deg2rad(out["SpinAxis"]))
    if "HorzBreak_m" not in out.columns and "HorzBreak" in out.columns:
        out["HorzBreak_m"] = out["HorzBreak"]
    if "RelSide_m" not in out.columns and "RelSide" in out.columns:
        out["RelSide_m"] = out["RelSide"]
    derived_inputs = {ex.GROUP_COL, ex.YEAR, ex.ALT, ex.PT, "RelSpeed", "SpinRate",
                      "InducedVertBreak", "HorzBreak_m", "RelHeight", "Extension"}
    if derived_inputs <= set(out.columns):
        out = ex.prepare_extra_features(out)
    else:
        if {"InducedVertBreak", "HorzBreak_m"} <= set(out.columns):
            out["total_break"] = np.sqrt(out["InducedVertBreak"] ** 2 + out["HorzBreak_m"] ** 2)
        if {"total_break", "SpinRate"} <= set(out.columns):
            out["move_per_rpm"] = out["total_break"] / out["SpinRate"].where(out["SpinRate"] > 0) * 1000
        if {"RelSpeed", "Extension", "RelHeight", "InducedVertBreak"} <= set(out.columns):
            out["vaa_est"] = ex._vaa_est(out)
    missing = [column for column in required if column not in out.columns]
    if missing:
        raise KeyError(f"No se pudieron reconstruir las features requeridas por FINAL_v1: {missing}")
    return out


def _load_model_artifact(paths: dict[str, Path], pitch_type: str) -> xgb.Booster:
    model_path = paths["root"] / "models" / f"stuff_{pitch_type}.json"
    if not model_path.exists() and "run_id" in paths:
        import mlflow
        from mlflow.tracking import MlflowClient
        client = MlflowClient()
        model_path = Path(client.download_artifacts(str(paths["run_id"]), f"models/stuff_{pitch_type}.json", str(paths["root"])))
    if not model_path.exists():
        raise FileNotFoundError(f"No hay modelo guardado para AutoPitchType={pitch_type!r}")
    model = xgb.Booster()
    model.load_model(str(model_path))
    return model


def _predict_in_batches(model: xgb.Booster, values: pd.DataFrame, batch_size: int = 50000) -> np.ndarray:
    """Predict in bounded batches so a competition dataframe does not monopolize the kernel."""
    predictions = []
    for start in range(0, len(values), batch_size):
        batch = values.iloc[start:start + batch_size]
        predictions.append(model.predict(xgb.DMatrix(batch)))
    return np.concatenate(predictions, axis=0)

def score_final_v1(frame: pd.DataFrame, run_name: str = RUN_NAME) -> pd.DataFrame:
    """Score rows with saved FINAL_v1 models and add Stuff_Plus_PitchType."""
    paths = _artifact_paths(run_name)
    config = json.loads(paths["config"].read_text(encoding="utf-8"))
    features = list(config["features_resolved"]) + list(config.get("controls", []))
    out = _prepare_run_features(frame, features)
    rv_table = pd.read_csv(paths["rv"], index_col=0).iloc[:, 0]
    classes = ex.SCHEMES[config["target"]]["classes"]
    rv = rv_table.reindex(classes).fillna(0.0).to_numpy(dtype="float32")
    oof = pd.read_parquet(paths["oof"])
    oof_pitch_column = ex.PT if ex.PT in oof.columns else "pitch_type"
    if not {oof_pitch_column, "xrv_use"} <= set(oof.columns):
        raise ValueError("El oof del run no contiene el tipo de pitcheo y xrv_use; no se puede calibrar Stuff+.")
    calibration = oof.groupby(oof_pitch_column)["xrv_use"].agg(["mean", "std"])
    out[SCORE_COLUMN] = np.nan
    if ex.PT not in out.columns:
        raise KeyError(f"Falta la columna de tipo de pitcheo requerida: {ex.PT}")

    for pitch_type in calibration.index:
        mask = out[ex.PT].eq(pitch_type)
        if not mask.any():
            continue
        model = _load_model_artifact(paths, str(pitch_type))
        X = out.loc[mask, features].astype("float32")
        if config.get("hand_neutral") and "same_hand" in features:
            predictions = []
            for hand in (0.0, 1.0):
                predictions.append(_predict_in_batches(model, X.assign(same_hand=hand)))
            probabilities = (predictions[0] + predictions[1]) / 2.0
        else:
            probabilities = _predict_in_batches(model, X)
        probabilities = probabilities.reshape(len(X), -1)
        xrv = probabilities @ rv
        mean = float(calibration.loc[pitch_type, "mean"])
        std = float(calibration.loc[pitch_type, "std"])
        if not np.isfinite(std) or std == 0:
            raise ValueError(f"Desviaci?n inv?lida en la calibraci?n de {pitch_type!r}: {std}")
        out.loc[mask, SCORE_COLUMN] = 100.0 - 10.0 * (xrv - mean) / std

    out["Stuff_Plus_PitchType_FINALv1"] = out[SCORE_COLUMN]
    return out


def rank_pitchers(frame: pd.DataFrame, metric: str = SCORE_COLUMN, min_pitches: int = 100) -> pd.DataFrame:
    """Return pitcher and pitch-type rankings with a minimum sample size."""
    required = {ex.GROUP_COL, ex.PT, metric}
    missing = required - set(frame.columns)
    if missing:
        raise KeyError(f"Faltan columnas para el ranking: {sorted(missing)}")
    return (
        frame.dropna(subset=[metric])
        .groupby([ex.GROUP_COL, ex.PT], as_index=False)
        .agg(stuff_plus=(metric, "mean"), n_pitches=(metric, "size"))
        .query("n_pitches >= @min_pitches")
        .sort_values([ex.PT, "stuff_plus"], ascending=[True, False])
    )


def plot_final_v1(frame: pd.DataFrame, metric: str = SCORE_COLUMN):
    """Plot score distributions without requiring location columns."""
    import matplotlib.pyplot as plt
    data = frame.dropna(subset=[metric, ex.PT])
    fig, ax = plt.subplots(figsize=(12, 5))
    data.boxplot(column=metric, by=ex.PT, ax=ax, grid=False, showfliers=False)
    ax.axhline(100, color="red", linestyle="--", alpha=0.6)
    ax.set_title(f"{metric} cargado desde {RUN_NAME}")
    ax.set_xlabel("Tipo de pitcheo")
    ax.set_ylabel("Stuff+")
    fig.suptitle("")
    fig.tight_layout()
    return fig


# Ejecuta ?nicamente inferencia y an?lisis sobre el dataframe ya cargado.
df = pd.read_parquet(INPUT_DATA_PATH)
df = score_final_v1(df)
rankings_final_v1 = rank_pitchers(df, min_pitches=100)
display(df[[ex.PT, SCORE_COLUMN, "Stuff_Plus_PitchType_FINALv1"]].head())
display(rankings_final_v1.head(30))
plot_final_v1(df)
